# 3D Gaussian Splatting (splatfacto) → SAM2-маска объекта → STL — чистый pipeline

Ядро Colab (Python 3.13 / torch 2.11) несовместимо с nerfstudio, поэтому nerfstudio ставится в отдельный venv `/content/nsenv` (Python 3.10), команды `ns-*` вызываются оттуда. Сегментация (Grounding DINO + SAM2) — прямо в ядре, ей нужен свежий torch.

`/content` эфемерный. После пересоздания рантайма — просто выполни все ячейки сверху вниз: готовые шаги сами пропустятся.

**Порядок:** GPU → setup venv (~10 мин, один раз) → датасет truck + фикс интринсик → сегментация SAM2 (~маски по тексту) → обучение с масками (~20 мин) → экспорт гауссиан → Poisson-mesh + DBSCAN → STL.

## 1. GPU и ядро

In [ ]:
import sys
print('kernel:', sys.version.split()[0])
!nvidia-smi -L

kernel: 3.13.15
GPU 0: Tesla T4 (UUID: GPU-cc735840-c1c8-b064-a68e-cc26279d8660)


## 2. Setup venv (Python 3.10 + torch 2.1.2 + gsplat 1.4.0 прекомпилированный + nerfstudio 1.1.5)

Одна ячейка, ~10 мин. Если venv уже есть — быстрая проверка и выход.

Ключевое: Python **3.10** (готовые колёса gsplat есть только под cp310), gsplat ставится прекомпилированным колесом `+pt21cu118` — никакой компиляции (иначе OOM-kill на RAM Colab). `setuptools<70` — из-за `pkg_resources.packaging`. `numpy==1.26.4` — иначе ломается импорт torch.

In [ ]:
%%bash
set -e
NSPY=/content/nsenv/bin/python

if [ -x /content/nsenv/bin/ns-train ]; then
  echo '=== venv уже собран, проверка ==='
  $NSPY -c "import numpy,torch,gsplat; from nerfstudio.models.splatfacto import SplatfactoModel; print('numpy',numpy.__version__,'| torch',torch.__version__,'| gsplat',gsplat.__version__,'| cuda',torch.cuda.is_available(),'| splatfacto OK')"
  exit 0
fi

pip install -q uv
uv python install 3.10
uv venv --python 3.10 /content/nsenv

uv pip install --python $NSPY "torch==2.1.2" "torchvision==0.16.2" --index-url https://download.pytorch.org/whl/cu118
uv pip install --python $NSPY "setuptools<70" wheel
uv pip install --python $NSPY "gsplat==1.4.0+pt21cu118" --index-url https://docs.gsplat.studio/whl/pt21cu118 --extra-index-url https://pypi.org/simple --index-strategy unsafe-best-match
uv pip install --python $NSPY "nerfstudio==1.1.5" open3d trimesh
uv pip install --python $NSPY "numpy==1.26.4"

echo '=== проверка ==='
$NSPY -c "import numpy,torch,gsplat; from gsplat.cuda._backend import _C; from nerfstudio.models.splatfacto import SplatfactoModel; print('numpy',numpy.__version__,'| torch',torch.__version__,'| gsplat',gsplat.__version__,'| cuda',torch.cuda.is_available(),'| splatfacto OK')"
/content/nsenv/bin/ns-train --help >/dev/null 2>&1 && echo 'ns-train: OK' || echo 'ns-train: FAIL'

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.4/20.4 MB 90.9 MB/s eta 0:00:00
=== проверка ===
numpy 1.26.4 | torch 2.1.2+cu118 | gsplat 1.4.0+pt21cu118 | cuda True | splatfacto OK
ns-train: OK


Installed Python 3.10.21 in 1.20s
 + cpython-3.10.21-linux-x86_64-gnu (python3.10)
Using CPython 3.10.21
Creating virtual environment at: nsenv
Activate with: source nsenv/bin/activate
Using Python 3.10.21 environment at: nsenv
Resolved 18 packages in 986ms
Prepared 18 packages in 40.53s
Installed 18 packages in 245ms
 + certifi==2022.12.7
 + charset-normalizer==2.1.1
 + filelock==3.32.3
 + fsspec==2026.7.0
 + idna==3.4
 + jinja2==3.1.6
 + markupsafe==3.0.3
 + mpmath==1.3.0
 + networkx==3.4.2
 + numpy==2.2.6
 + pillow==12.3.0
 + requests==2.28.1
 + sympy==1.14.0
 + torch==2.1.2+cu118
 + torchvision==0.16.2+cu118
 + triton==2.1.0
 + typing-extensions==4.16.0
 + urllib3==1.26.13
Using Python 3.10.21 environment at: nsenv
Resolved 3 packages in 144ms
Prepared 3 packages in 71ms
Installed 3 packages in 14ms
 + packaging==26.3
 + setuptools==69.5.1
 + wheel==0.48.0
Using Python 3.10.21 environment at: nsenv
Resolved 19 packages in 1.55s
Prepared 8 packages in 1.01s
Installed 8 packages in 3

## 3. Датасет truck + фикс cameras.bin

Картинки в `images/` уже уменьшены (979×546), а `cameras.bin` хранит интринсики полного размера (1957×1091) → масштабируем fx/fy/cx/cy. Оригинал → `cameras.bin.orig`.

In [ ]:
import os, subprocess

D = '/content/tandt/truck'
if not os.path.isdir(f'{D}/images'):
    os.system('wget -q --show-progress https://repo-sam.inria.fr/fungraph/3d-gaussian-splatting/datasets/input/tandt_db.zip -O /content/tandt_db.zip')
    os.system('cd /content && unzip -q -o tandt_db.zip')
print('кадров:', len(os.listdir(f'{D}/images')))
print('COLMAP:', sorted(os.listdir(f'{D}/sparse/0')))

if not os.path.exists(f'{D}/sparse/0/cameras.bin.orig'):
    code = r'''
import glob, shutil
from PIL import Image
from nerfstudio.data.utils.colmap_parsing_utils import read_cameras_binary, write_cameras_binary, Camera
sp, im = "/content/tandt/truck/sparse/0", "/content/tandt/truck/images"
cams = read_cameras_binary(f"{sp}/cameras.bin")
w, h = Image.open(sorted(glob.glob(im + "/*"))[0]).size
new = {}
for k, c in cams.items():
    sx, sy = w / c.width, h / c.height
    p = list(c.params)
    if c.model == "PINHOLE":
        p = [p[0]*sx, p[1]*sy, p[2]*sx, p[3]*sy]
    elif c.model in ("SIMPLE_PINHOLE", "SIMPLE_RADIAL"):
        p[0] *= sx; p[1] *= sx; p[2] *= sy
    else:
        raise SystemExit("модель %s не поддержана" % c.model)
    new[k] = Camera(id=c.id, model=c.model, width=w, height=h, params=p)
shutil.copy(f"{sp}/cameras.bin", f"{sp}/cameras.bin.orig")
write_cameras_binary(new, f"{sp}/cameras.bin")
[print("cameras.bin →", v.width, v.height, [round(x, 2) for x in v.params]) for v in read_cameras_binary(f"{sp}/cameras.bin").values()]
'''
    r = subprocess.run(['/content/nsenv/bin/python', '-c', code], capture_output=True, text=True)
    print(r.stdout or r.stderr[-2000:])
else:
    print('cameras.bin уже пофикшен')

кадров: 251
COLMAP: ['cameras.bin', 'images.bin', 'points3D.bin', 'project.ini']
cameras.bin → 979 546 [581.92, 578.67, 489.5, 273.0]



## 4. Сегментация: текст → маска (Grounding DINO + SAM2)

Маска объекта по текстовому промпту (`OBJECT_PROMPT`) строится ДО обучения — фоновые гауссианы никогда не учатся. Это меняет **SAM v1 → SAM2** (`facebook/sam2-hiera-large`, HF `transformers` `Sam2Model`/`Sam2Processor`) — Grounding DINO не трогаем. Работает в ЯДРЕ Colab (Python 3.13/torch 2.11), не в venv.</cell id="cell-6">

In [ ]:
!pip install -q -U transformers
import torch, numpy as np
from PIL import Image
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection, Sam2Model, Sam2Processor

dev = "cuda"
OBJECT_PROMPT = "a truck."  # текстовое описание объекта пользователя — заменить под реальный сценарий
GD, SAM = "IDEA-Research/grounding-dino-base", "facebook/sam2-hiera-large"
gd_proc = AutoProcessor.from_pretrained(GD)
gd_model = AutoModelForZeroShotObjectDetection.from_pretrained(GD).to(dev).eval()
sam_proc = Sam2Processor.from_pretrained(SAM)
sam_model = Sam2Model.from_pretrained(SAM).to(dev).eval()
print("модели ок (SAM2)")

def gdino_boxes(img, text=OBJECT_PROMPT, thr=0.3):
    inp = gd_proc(images=img, text=text, return_tensors="pt").to(dev)
    with torch.no_grad():
        out = gd_model(**inp)
    tgt = [img.size[::-1]]
    try:
        r = gd_proc.post_process_grounded_object_detection(out, inp.input_ids, threshold=thr, text_threshold=0.25, target_sizes=tgt)[0]
    except TypeError:
        r = gd_proc.post_process_grounded_object_detection(out, inp.input_ids, box_threshold=thr, text_threshold=0.25, target_sizes=tgt)[0]
    return r["boxes"].cpu().numpy(), r["scores"].cpu().numpy()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 102.5 MB/s eta 0:00:0000:010:01


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


preprocessor_config.json:   0%|          | 0.00/457 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.74k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.24k [00:00<?, ?B/s]

...

[transformers] You are using a model of type `sam2_video` to instantiate a model of type `sam2`. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.


model.safetensors: reconstructing file:   0%|          |  0.00B /  898MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/741 [00:00<?, ?it/s]

модели ок (SAM2)


In [ ]:
import os
from tqdm import tqdm

IMG_DIR  = f'{D}/images'
MASK_DIR = f'{D}/masks'
PREV_DIR = f'{D}/_mask_preview'
os.makedirs(MASK_DIR, exist_ok=True); os.makedirs(PREV_DIR, exist_ok=True)

_pp = getattr(sam_proc, "post_process_masks", None) or sam_proc.image_processor.post_process_masks

def sam_union(img, boxes):
    inp = sam_proc(img, input_boxes=[[list(map(float, b)) for b in boxes]], return_tensors="pt").to(dev)
    with torch.no_grad():
        out = sam_model(**inp, multimask_output=True)
    masks = _pp(out.pred_masks.cpu(), inp["original_sizes"].cpu())[0]
    m = masks.numpy()
    iou = out.iou_scores.cpu().numpy()[0]
    sel = np.stack([m[i, iou[i].argmax()] for i in range(m.shape[0])])
    return sel.any(0)

names = sorted(os.listdir(IMG_DIR))
no_box = []
if os.path.isdir(MASK_DIR) and len(os.listdir(MASK_DIR)) == len(names):
    print('маски уже есть:', len(names))
else:
    for i, n in enumerate(tqdm(names)):
        img = Image.open(f"{IMG_DIR}/{n}").convert("RGB")
        W, H = img.size
        b, s = gdino_boxes(img, OBJECT_PROMPT, 0.3)
        if len(b) == 0:
            b, s = gdino_boxes(img, "объект. object. vehicle.", 0.22)
        keep = [bb for bb in b if 0.02 < ((bb[2]-bb[0])*(bb[3]-bb[1]))/(W*H) < 0.99]
        if not keep:
            no_box.append(n)
            mask = np.ones((H, W), bool)
        else:
            mask = sam_union(img, keep)
        Image.fromarray((mask*255).astype(np.uint8)).save(f"{MASK_DIR}/{os.path.splitext(n)[0]}.png")
        if i % 42 == 0:
            ov = np.array(img); ov[~mask] = (ov[~mask]*0.2).astype(np.uint8)
            Image.fromarray(ov).save(f"{PREV_DIR}/{n}")
    print("масок:", len(os.listdir(MASK_DIR)), "| без бокса:", len(no_box), no_box[:10])

100%|██████████| 251/251 [06:26<00:00,  1.54s/it]

масок: 251 | без бокса: 0 []


## 5. Обучение splatfacto с масками (только объект, без фона)

~20 мин на T4. Маски из шага 4 передаются как `--masks-path masks` — фон никогда не учится. Чекпойнты каждые 2000 шагов в `outputs_masked/truck/splatfacto/<ts>/nerfstudio_models/`.

Если `CUDA out of memory` — поставь `--downscale-factor 2` и `--pipeline.model.num-downscales 2`.

In [ ]:
!yes | TORCHDYNAMO_DISABLE=1 /content/nsenv/bin/ns-train splatfacto --data /content/tandt/truck --max-num-iterations 7000 --steps-per-save 2000 --vis tensorboard --viewer.quit-on-train-completion True --output-dir outputs_masked --pipeline.model.cull-alpha-thresh 0.1 colmap --images-path images --colmap-path sparse/0 --masks-path masks --downscale-factor 1

[16:14:11] Using --data alias for --data.pipeline.datamanager.data                                          ]8;id=14649437;file:///content/nsenv/lib/python3.10/site-packages/nerfstudio/scripts/train.py\train.py]8;;\:]8;id=14649438;file:///content/nsenv/lib/python3.10/site-packages/nerfstudio/scripts/train.py#230\230]8;;\
──────────────────────────────────────────────────────── Config ────────────────────────────────────────────────────────
TrainerConfig(
    _target=<class 'nerfstudio.engine.trainer.Trainer'>,
    output_dir=PosixPath('outputs_masked'),
    method_name='splatfacto',
    experiment_name=None,
    project_name='nerfstudio-project',
    timestamp='2026-09-30_161411',
    machine=MachineConfig(seed=42, num_devices=1, num_machines=1, machine_rank=0, dist_url='auto', device_type='cuda'),
    logging=LoggingConfig(
        relative_log_dir=PosixPath('.'),
        steps_per_log=10,
        max_buffer_size=20,
        local_writer=LocalWriterConfig(
            _target=

## 6. Путь к конфигу обученной модели

In [ ]:
import glob, os
cfgs = sorted(glob.glob('outputs_masked/truck/splatfacto/*/config.yml'), key=os.path.getmtime)
assert cfgs, 'config.yml нет — обучение не завершилось'
CONFIG = cfgs[-1]
print('CONFIG =', CONFIG)

CONFIG = outputs_masked/truck/splatfacto/2026-09-30_161411/config.yml


## 7. Экспорт гауссиан → .ply

In [ ]:
!/content/nsenv/bin/ns-export gaussian-splat --load-config {CONFIG} --output-dir exports/truck_masked
!ls -la exports/truck_masked

Loading latest checkpoint from load_dir
✅ Done loading checkpoint from outputs_masked/truck/splatfacto/2026-09-30_161411/nerfstudio_models/step-000006999.ckpt
0 Gaussians have NaN/Inf and 3 have low opacity, only export 120482/120485
total 29192
drwxr-xr-x 2 root root     4096 Sep 30 16:18 .
drwxr-xr-x 3 root root     4096 Sep 30 16:18 ..
-rw-r--r-- 1 root root 29881129 Sep 30 16:18 splat.ply


## 8. Poisson-mesh → STL (только объект)

Облако гауссиан (уже без фона благодаря маске) → чистка выбросов → DBSCAN, оставляем крупнейший кластер → нормали → Screened Poisson → STL. Низ объекта, скорее всего, не отснят — меш может быть не watertight (руками в Blender/Meshmixer при необходимости печати).

In [ ]:
script = r'''
import glob
import numpy as np
import open3d as o3d
import trimesh

ply = sorted(glob.glob("exports/truck_masked/*.ply"))[-1]
print("PLY:", ply)
pcd = o3d.io.read_point_cloud(ply)
print("Punkte:", len(pcd.points))
pcd, _ = pcd.remove_statistical_outlier(nb_neighbors=20, std_ratio=2.0)
print("nach Bereinigung:", len(pcd.points))

nn = o3d.geometry.KDTreeFlann(pcd)
step = max(1, len(pcd.points) // 2000)
d = []
for i in range(0, len(pcd.points), step):
    _, idx, dist2 = nn.search_knn_vector_3d(pcd.points[i], 2)
    d.append(np.sqrt(dist2[1]))
eps = 5 * float(np.median(d))
labels = np.array(pcd.cluster_dbscan(eps=eps, min_points=10))
if labels.max() >= 0:
    biggest = np.bincount(labels[labels >= 0]).argmax()
    pcd = pcd.select_by_index(np.where(labels == biggest)[0])
print("nach DBSCAN (groesster Cluster):", len(pcd.points))

pcd.estimate_normals(o3d.geometry.KDTreeSearchParamHybrid(radius=0.1, max_nn=30))
pcd.orient_normals_consistent_tangent_plane(30)
mesh, dens = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(pcd, depth=10)
dens = np.asarray(dens)
mesh.remove_vertices_by_mask(dens <= np.quantile(dens, 0.1))
mesh.remove_degenerate_triangles(); mesh.remove_duplicated_vertices()
mesh.remove_duplicated_triangles(); mesh.remove_non_manifold_edges()
mesh.compute_triangle_normals()
out = "exports/truck_masked/truck_object.stl"
o3d.io.write_triangle_mesh(out, mesh)

m = trimesh.load(out)
m.fill_holes(); m.fix_normals(); m.process(validate=True)
m.export(out)
print("STL:", out, "| Ecken:", len(m.vertices), "| Dreiecke:", len(m.faces))
print("watertight:", m.is_watertight, "| bbox:", [round(float(x), 3) for x in m.bounding_box.extents])
'''
open('/content/mesh_to_stl.py', 'w').write(script)
import subprocess
r = subprocess.run(['/content/nsenv/bin/python', '/content/mesh_to_stl.py'], capture_output=True, text=True)
print(r.stdout)
if r.returncode:
    print('STDERR:\n', r.stderr[-3000:])

PLY: exports/truck_masked/splat.ply
Punkte: 120482
nach Bereinigung: 117897
nach DBSCAN (groesster Cluster): 77364
STL: exports/truck_masked/truck_object.stl | Ecken: 219250 | Dreiecke: 425112
watertight: False | bbox: [1.06, 0.422, 0.355]



## Результат

`exports/truck_masked/truck_object.stl` — только объект (фон срезан маской SAM2+DINO). Взять через панель файлов VS Code (`content/exports/truck_masked/` → правый клик → Download) или скопировать на Drive.

In [ ]:
!apt-get -qq install -y colmap
!colmap -h | head -5


E: Failed to fetch http://archive.ubuntu.com/ubuntu/pool/main/libi/libinput/libinput-bin_1.25.0-1ubuntu3.6_amd64.deb  404  Not Found [IP: 91.189.91.83 80]
E: Failed to fetch http://archive.ubuntu.com/ubuntu/pool/main/libi/libinput/libinput10_1.25.0-1ubuntu3.6_amd64.deb  404  Not Found [IP: 91.189.91.83 80]
E: Unable to fetch some archives, maybe run apt-get update or try with --fix-missing?
/bin/bash: line 1: colmap: command not found


In [ ]:
import os, shutil, random

SRC = f'{D}/images'
RAW = '/content/sparse_test/raw'
os.makedirs(RAW, exist_ok=True)

names = sorted(os.listdir(SRC))
random.seed(0)
subset = sorted(random.sample(names, 20))
for n in subset:
    shutil.copy(f'{SRC}/{n}', f'{RAW}/{n}')

print('скопировано:', len(os.listdir(RAW)))


скопировано: 20


In [ ]:
import os, shutil, random

SRC = '/content/tandt/truck/images'
RAW = '/content/sparse_test/raw'
os.makedirs(RAW, exist_ok=True)

names = sorted(os.listdir(SRC))
random.seed(0)
subset = sorted(random.sample(names, 20))
for n in subset:
    shutil.copy(f'{SRC}/{n}', f'{RAW}/{n}')

print('скопировано:', len(os.listdir(RAW)))


скопировано: 20


In [ ]:
!nvidia-smi -L


GPU 0: Tesla T4 (UUID: GPU-cc735840-c1c8-b064-a68e-cc26279d8660)


In [ ]:
img_test = Image.open(f"{IMG_DIR}/{names[0]}").convert("RGB")
inp_test = sam_proc(img_test, input_boxes=[[[0,0,100,100]]], return_tensors="pt").to(dev)
print(list(inp_test.keys()))

import inspect
print(inspect.signature(_pp))


['pixel_values', 'original_sizes', 'input_boxes']
(masks, original_sizes, mask_threshold=0.0, binarize=True, max_hole_area=0.0, max_sprinkle_area=0.0, apply_non_overlapping_constraints=False, **kwargs)


In [ ]:
script = r'''
import glob
import numpy as np
import open3d as o3d
import trimesh

ply = sorted(glob.glob("exports/truck_masked/*.ply"))[-1]
print("PLY:", ply)
pcd = o3d.io.read_point_cloud(ply)
print("Punkte:", len(pcd.points))
pcd, _ = pcd.remove_statistical_outlier(nb_neighbors=20, std_ratio=2.0)
print("nach Bereinigung:", len(pcd.points))

nn = o3d.geometry.KDTreeFlann(pcd)
step = max(1, len(pcd.points) // 2000)
d = []
for i in range(0, len(pcd.points), step):
    _, idx, dist2 = nn.search_knn_vector_3d(pcd.points[i], 2)
    d.append(np.sqrt(dist2[1]))
eps = 5 * float(np.median(d))
labels = np.array(pcd.cluster_dbscan(eps=eps, min_points=10))
if labels.max() >= 0:
    biggest = np.bincount(labels[labels >= 0]).argmax()
    pcd = pcd.select_by_index(np.where(labels == biggest)[0])
print("nach DBSCAN (groesster Cluster):", len(pcd.points))

pcd.estimate_normals(o3d.geometry.KDTreeSearchParamHybrid(radius=0.1, max_nn=30))
pcd.orient_normals_consistent_tangent_plane(30)
mesh, dens = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(pcd, depth=10)
dens = np.asarray(dens)
mesh.remove_vertices_by_mask(dens <= np.quantile(dens, 0.1))
mesh.remove_degenerate_triangles(); mesh.remove_duplicated_vertices()
mesh.remove_duplicated_triangles(); mesh.remove_non_manifold_edges()
mesh.compute_triangle_normals()
out = "exports/truck_masked/truck_object.stl"
o3d.io.write_triangle_mesh(out, mesh)

m = trimesh.load(out)
m.fill_holes(); m.fix_normals(); m.process(validate=True)
m.export(out)
print("STL:", out, "| Ecken:", len(m.vertices), "| Dreiecke:", len(m.faces))
print("watertight:", m.is_watertight, "| bbox:", [round(float(x), 3) for x in m.bounding_box.extents])
'''
open('/content/mesh_to_stl.py', 'w').write(script)
import subprocess
r = subprocess.run(['/content/nsenv/bin/python', '/content/mesh_to_stl.py'], capture_output=True, text=True)
print(r.stdout)
if r.returncode:
    print('STDERR:\n', r.stderr[-3000:])


PLY: exports/truck_masked/splat.ply
Punkte: 120482
nach Bereinigung: 117897
nach DBSCAN (groesster Cluster): 77364
STL: exports/truck_masked/truck_object.stl | Ecken: 219251 | Dreiecke: 425144
watertight: False | bbox: [1.06, 0.422, 0.355]



In [ ]:
!/content/nsenv/bin/ns-process-data images \
  --data /content/sparse_test/raw \
  --output-dir /content/sparse_test/processed \
  --matching-method exhaustive


Could not find COLMAP. Please install COLMAP.
See https://colmap.github.io/install.html for installation instructions.


In [ ]:
!rm -rf /content/sparse_test/processed
!QT_QPA_PLATFORM=offscreen /content/nsenv/bin/ns-process-data images \
  --data /content/sparse_test/raw \
  --output-dir /content/sparse_test/processed \
  --matching-method exhaustive


Could not find COLMAP. Please install COLMAP.
See https://colmap.github.io/install.html for installation instructions.


In [ ]:
import os, shutil, subprocess

N = 12
SRC = "/content/tandt/truck"
DST = f"/content/tandt/truck_sparse{N}"
os.makedirs(f"{DST}/images", exist_ok=True)
os.makedirs(f"{DST}/sparse/0", exist_ok=True)

names = sorted(os.listdir(f"{SRC}/images"))
step = len(names) / N
chosen = [names[int(i * step)] for i in range(N)]
print("выбрано кадров:", len(chosen))
print(chosen)

for n in chosen:
    shutil.copy(f"{SRC}/images/{n}", f"{DST}/images/{n}")
shutil.copy(f"{SRC}/sparse/0/cameras.bin", f"{DST}/sparse/0/cameras.bin")
shutil.copy(f"{SRC}/sparse/0/project.ini", f"{DST}/sparse/0/project.ini")

code = f'''
import numpy as np
from nerfstudio.data.utils.colmap_parsing_utils import (
    read_images_binary, write_images_binary,
    read_points3D_binary, write_points3D_binary)

chosen = {chosen!r}
imgs = read_images_binary("{SRC}/sparse/0/images.bin")
kept = {{k: v for k, v in imgs.items() if v.name in chosen}}
print("оставлено камер:", len(kept), "из", len(imgs))
write_images_binary(kept, "{DST}/sparse/0/images.bin")

kept_ids = set(kept.keys())
pts = read_points3D_binary("{SRC}/sparse/0/points3D.bin")
pts_kept = {{}}
for pid, p in pts.items():
    mask = np.array([i for i, imid in enumerate(p.image_ids) if imid in kept_ids], dtype=int)
    if mask.size == 0:
        continue
    pts_kept[pid] = p._replace(image_ids=p.image_ids[mask], point2D_idxs=p.point2D_idxs[mask])
print("оставлено 3D-точек:", len(pts_kept), "из", len(pts))
write_points3D_binary(pts_kept, "{DST}/sparse/0/points3D.bin")
'''
r = subprocess.run(['/content/nsenv/bin/python', '-c', code], capture_output=True, text=True)
print(r.stdout)
if r.returncode:
    print("STDERR:\n", r.stderr[-3000:])


выбрано кадров: 12
['000001.jpg', '000021.jpg', '000042.jpg', '000063.jpg', '000084.jpg', '000105.jpg', '000126.jpg', '000147.jpg', '000168.jpg', '000189.jpg', '000210.jpg', '000231.jpg']
оставлено камер: 12 из 251
оставлено 3D-точек: 40991 из 136029



In [ ]:
!/content/nsenv/bin/ns-process-data images \
  --data /content/sparse_test/raw \
  --output-dir /content/sparse_test/processed \
  --matching-method exhaustive


Could not find COLMAP. Please install COLMAP.
See https://colmap.github.io/install.html for installation instructions.


In [ ]:
!rm -rf /content/sparse_test/processed
!QT_QPA_PLATFORM=offscreen /content/nsenv/bin/ns-process-data images \
  --data /content/sparse_test/raw \
  --output-dir /content/sparse_test/processed \
  --matching-method exhaustive


Could not find COLMAP. Please install COLMAP.
See https://colmap.github.io/install.html for installation instructions.


In [ ]:
!rm -rf /content/sparse_test/processed
!QT_QPA_PLATFORM=offscreen /content/nsenv/bin/ns-process-data images \
  --data /content/sparse_test/raw \
  --output-dir /content/sparse_test/processed \
  --matching-method exhaustive \
  --no-gpu


Could not find COLMAP. Please install COLMAP.
See https://colmap.github.io/install.html for installation instructions.


In [ ]:
!rm -rf /content/sparse_test/processed
!QT_QPA_PLATFORM=offscreen /content/nsenv/bin/ns-process-data images \
  --data /content/sparse_test/raw \
  --output-dir /content/sparse_test/processed \
  --matching-method exhaustive \
  --no-gpu

Could not find COLMAP. Please install COLMAP.
See https://colmap.github.io/install.html for installation instructions.


In [ ]:
!rm -rf /content/sparse_test/processed
!QT_QPA_PLATFORM=offscreen /content/nsenv/bin/ns-process-data images \
  --data /content/sparse_test/raw \
  --output-dir /content/sparse_test/processed \
  --matching-method exhaustive \
  --no-gpu


Could not find COLMAP. Please install COLMAP.
See https://colmap.github.io/install.html for installation instructions.


In [ ]:
!ls /content/nsenv/bin/ 2>&1 | head -3


activate
activate.bat
activate.csh


In [ ]:
import os
os.makedirs('/content/drive/MyDrive/nerf_cache', exist_ok=True)
CACHE = '/content/drive/MyDrive/nerf_cache/content_cache.tar.gz'
if not os.path.exists(CACHE):
    print('Архивирую и заливаю на Drive (может занять пару минут)...')
    os.system(f'tar -czf {CACHE} /content/nsenv /content/tandt')
    print('Готово, размер:', round(os.path.getsize(CACHE)/1e9, 2), 'GB')
else:
    print('Кэш уже есть на Drive.')


Архивирую и заливаю на Drive (может занять пару минут)...


In [ ]:
import os
print('venv:', os.path.exists('/content/nsenv/bin/ns-train'))
print('датасет:', os.path.exists('/content/tandt/truck/images'))


venv: True
датасет: True


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
CACHE = '/content/drive/MyDrive/nerf_cache/content_cache.tar.gz'
if os.path.exists(CACHE):
    print('Кэш найден, восстанавливаю...')
    os.system(f'tar -xzf {CACHE} -C /')
    print('venv:', os.path.exists('/content/nsenv/bin/ns-train'), '| маски:', os.path.exists('/content/tandt/truck/masks'))
else:
    print('Кэша нет!')


MessageError: Failed to issue request GET https://colab.research.google.com/tun/m/credentials-propagation/gpu-t4-s-kkb-usw1b2-1ljejrke53gzh?authtype=dfs_ephemeral&version=2&dryrun=true&propagate=true&record=false&authuser=0: Not Found
Response body: 
<!DOCTYPE html>
<html lang=en>
  <meta charset=utf-8>
  <meta name=viewport content="initial-scale=1, minimum-scale=1, width=device-width">
  <title>Error 404 (Not Found)!!1</title>
  <style>
    *{margin:0;padding:0}html,code{font:15px/22px arial,sans-serif}html{background:#fff;color:#222;padding:15px}body{margin:7% auto 0;max-width:390px;min-height:180px;padding:30px 0 15px}* > body{background:url(//www.google.com/images/errors/robot.png) 100% 5px no-repeat;padding-right:205px}p{margin:11px 0 22px;overflow:hidden}ins{color:#777;text-decoration:none}a img{border:0}@media screen and (max-width:772px){body{background:none;margin-top:0;max-width:none;padding-right:0}}#logo{background:url(//www.google.com/images/logos/errorpage/error_logo-150

In [ ]:
import os
d = '/content/drive/MyDrive/nerf_cache'
print(os.path.exists(d))
print(os.listdir(d) if os.path.exists(d) else 'папки нет')
print(os.listdir('/content/drive/MyDrive'))


False
папки нет
['Інформатика - підручник для 7 класу  авт.Ривкінд Й.Я.  та ін..PDF', 'Справка qt.gdoc', 'овдиенко 9в.docx.gdoc', 'Новый рисунок.gdraw', 'Новый документ.gdoc', '7 Obwody pr_du sinusoidalnego 98-130.rar', '6C22A32B-A312-4BA5-ABEE-5A2F4F3048AD.jpeg', 'images', 'Eie', 'Classroom', 'Eng', 'ppih.zz.docx', 'kr.docx', '22 мая 2020 г., 10:13 (1)', '22 мая 2020 г., 10:13', '22 мая 2020 г., 10:46', 'kar.docx', 'kar.gdoc', 'кр последняя.docx', 'pass (2).pkpass', 'pol', 'mull', 'ANGLhodfsfdsfsdfsdfsdfsdfsdfsd (1).docx', 'ANGLhodfsfdsfsdfsdfsdfsdfsdfsd.docx', 'Резюме.gdoc', 'job offer.gdoc', 'ProjectTiaH.zip', 'ProjectSimulationH24.vcmx', 'Colab Notebooks', 'ProjectVideo', 'nerf_day1', 'nerf_day2', 'nerf_day2_30k']


In [ ]:
%%bash
set -e
NSPY=/content/nsenv/bin/python

if [ -x /content/nsenv/bin/ns-train ]; then
  echo '=== venv bereits eingerichtet ==='
  exit 0
fi

pip install -q uv
uv python install 3.10
uv venv --python 3.10 /content/nsenv

uv pip install --python $NSPY "torch==2.1.2" "torchvision==0.16.2" --index-url https://download.pytorch.org/whl/cu118
uv pip install --python $NSPY "setuptools<70" wheel
uv pip install --python $NSPY "gsplat==1.4.0+pt21cu118" --index-url https://docs.gsplat.studio/whl/pt21cu118 --extra-index-url https://pypi.org/simple --index-strategy unsafe-best-match
uv pip install --python $NSPY "nerfstudio==1.1.5" open3d trimesh
uv pip install --python $NSPY "numpy==1.26.4"

$NSPY -c "import numpy,torch,gsplat; from nerfstudio.models.splatfacto import SplatfactoModel; print('numpy',numpy.__version__,'| torch',torch.__version__,'| gsplat',gsplat.__version__,'| cuda',torch.cuda.is_available(),'| splatfacto OK')"


=== venv bereits eingerichtet ===


In [ ]:
import os

D = '/content/tandt/truck'
if not os.path.isdir(f'{D}/images'):
    os.system('wget -q --show-progress https://repo-sam.inria.fr/fungraph/3d-gaussian-splatting/datasets/input/tandt_db.zip -O /content/tandt_db.zip')
    os.system('cd /content && unzip -q -o tandt_db.zip')
print('кадров:', len(os.listdir(f'{D}/images')))


кадров: 251


In [ ]:
!apt-get -qq install -y colmap


E: Failed to fetch http://archive.ubuntu.com/ubuntu/pool/main/libi/libinput/libinput-bin_1.25.0-1ubuntu3.6_amd64.deb  404  Not Found [IP: 185.125.190.81 80]
E: Failed to fetch http://archive.ubuntu.com/ubuntu/pool/main/libi/libinput/libinput10_1.25.0-1ubuntu3.6_amd64.deb  404  Not Found [IP: 185.125.190.81 80]
E: Unable to fetch some archives, maybe run apt-get update or try with --fix-missing?


In [ ]:
!apt-get -qq update
!apt-get -qq install -y colmap


W: https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt/trusted.gpg), see the DEPRECATION section in apt-key(8) for details.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Extracting templates from packages: 100%
Selecting previously unselected package libevdev2:amd64.
(Reading database ... 126952 files and directories currently installed.)
Preparing to unpack .../00-libevdev2_1.13.1+dfsg-1build1_amd64.deb ...
Unpacking libevdev2:amd64 (1.13.1+dfsg-1build1) ...
Selecting previously unselected package at-spi2-common.
Preparing to unpack .../01-at-spi2-common_2.52.0-1build1_all.deb ...
Unpacking at-spi2-common (2.52.0-1build1) ...
Selecting previously unselected package libatspi2.0-0t64:amd64.
Preparing to unpack .../02-libatspi2.0-0t64_2.52.0-1build1_amd64.deb .

In [ ]:
!colmap -h | head -3


COLMAP 3.9.1 -- Structure-from-Motion and Multi-View Stereo
(Commit Unknown on Unknown without CUDA)



In [ ]:
import os, shutil, random

SRC = '/content/tandt/truck/images'
RAW = '/content/sparse_test30/raw'
os.makedirs(RAW, exist_ok=True)

names = sorted(os.listdir(SRC))
random.seed(0)
subset = sorted(random.sample(names, 30))
for n in subset:
    shutil.copy(f'{SRC}/{n}', f'{RAW}/{n}')

print('скопировано:', len(os.listdir(RAW)))


скопировано: 30


In [ ]:
!QT_QPA_PLATFORM=offscreen /content/nsenv/bin/ns-process-data images \
  --data /content/sparse_test30/raw \
  --output-dir /content/sparse_test30/processed \
  --matching-method exhaustive \
  --no-gpu


[09:03:53] 🎉 Done copying images with prefix 'frame_'.                                        ]8;id=10324174;file:///content/nsenv/lib/python3.10/site-packages/nerfstudio/process_data/process_data_utils.py\process_data_utils.py]8;;\:]8;id=10324175;file:///content/nsenv/lib/python3.10/site-packages/nerfstudio/process_data/process_data_utils.py#348\348]8;;\
( ●    ) Copying images...
🌘  Running COLMAP feature extractor...tor...
[09:04:38] 🎉 Done extracting COLMAP features.                                                       ]8;id=10324182;file:///content/nsenv/lib/python3.10/site-packages/nerfstudio/process_data/colmap_utils.py\colmap_utils.py]8;;\:]8;id=10324183;file:///content/nsenv/lib/python3.10/site-packages/nerfstudio/process_data/colmap_utils.py#137\137]8;;\
🚶  Running COLMAP feature matcher...0m
[09:08:43] 🎉 Done matching COLMAP features.                                                         ]8;id=10324189;file:///content/nsenv/lib/python3.10/site-packages/ne

In [ ]:
import os
print('venv:', os.path.exists('/content/nsenv/bin/ns-train'))
print('sparse20:', os.path.exists('/content/sparse_test/processed/transforms.json'))
print('sparse30:', os.path.exists('/content/sparse_test30/processed/transforms.json'))
print('полный STL:', os.path.exists('/content/exports/truck_masked/truck_object.stl'))


venv: True
sparse20: False
sparse30: True
полный STL: False


In [ ]:
import os

def check(p):
    return os.path.exists(p)

print('colmap:', os.system('which colmap') == 0)
print('sparse20 raw:', check('/content/sparse_test/raw'), len(os.listdir('/content/sparse_test/raw')) if check('/content/sparse_test/raw') else 0)
print('sparse30 processed images:', len(os.listdir('/content/sparse_test30/processed/images')) if check('/content/sparse_test30/processed/images') else 'нет')
print('full ply:', check('/content/exports/truck_masked/splat.ply'))
print('full masks:', len(os.listdir('/content/tandt/truck/masks')) if check('/content/tandt/truck/masks') else 'нет')
print('training outputs:', os.listdir('/content/outputs_masked/truck/splatfacto') if check('/content/outputs_masked/truck/splatfacto') else 'нет')


colmap: True
sparse20 raw: False 0
sparse30 processed images: 30
full ply: False
full masks: нет
training outputs: нет


In [ ]:
import os

D = '/content/tandt/truck'
if not os.path.isdir(f'{D}/images'):
    os.system('wget -q --show-progress https://repo-sam.inria.fr/fungraph/3d-gaussian-splatting/datasets/input/tandt_db.zip -O /content/tandt_db.zip')
    os.system('cd /content && unzip -q -o tandt_db.zip')
print('кадров:', len(os.listdir(f'{D}/images')))


кадров: 251


In [ ]:
!pip install -q -U transformers
import torch, numpy as np
from PIL import Image
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection, Sam2Model, Sam2Processor

dev = "cuda"
OBJECT_PROMPT = "a truck."
GD, SAM = "IDEA-Research/grounding-dino-base", "facebook/sam2-hiera-large"
gd_proc = AutoProcessor.from_pretrained(GD)
gd_model = AutoModelForZeroShotObjectDetection.from_pretrained(GD).to(dev).eval()
sam_proc = Sam2Processor.from_pretrained(SAM)
sam_model = Sam2Model.from_pretrained(SAM).to(dev).eval()
print("модели ок (SAM2)")

def gdino_boxes(img, text=OBJECT_PROMPT, thr=0.3):
    inp = gd_proc(images=img, text=text, return_tensors="pt").to(dev)
    with torch.no_grad():
        out = gd_model(**inp)
    tgt = [img.size[::-1]]
    try:
        r = gd_proc.post_process_grounded_object_detection(out, inp.input_ids, threshold=thr, text_threshold=0.25, target_sizes=tgt)[0]
    except TypeError:
        r = gd_proc.post_process_grounded_object_detection(out, inp.input_ids, box_threshold=thr, text_threshold=0.25, target_sizes=tgt)[0]
    return r["boxes"].cpu().numpy(), r["scores"].cpu().numpy()


: 

In [ ]:
!pip install -q -U transformers
import torch, numpy as np
from PIL import Image
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection, Sam2Model, Sam2Processor

dev = "cuda"
OBJECT_PROMPT = "a truck."
GD, SAM = "IDEA-Research/grounding-dino-base", "facebook/sam2-hiera-large"
gd_proc = AutoProcessor.from_pretrained(GD)
gd_model = AutoModelForZeroShotObjectDetection.from_pretrained(GD).to(dev).eval()
sam_proc = Sam2Processor.from_pretrained(SAM)
sam_model = Sam2Model.from_pretrained(SAM).to(dev).eval()
print("модели ок (SAM2)")

def gdino_boxes(img, text=OBJECT_PROMPT, thr=0.3):
    inp = gd_proc(images=img, text=text, return_tensors="pt").to(dev)
    with torch.no_grad():
        out = gd_model(**inp)
    tgt = [img.size[::-1]]
    try:
        r = gd_proc.post_process_grounded_object_detection(out, inp.input_ids, threshold=thr, text_threshold=0.25, target_sizes=tgt)[0]
    except TypeError:
        r = gd_proc.post_process_grounded_object_detection(out, inp.input_ids, box_threshold=thr, text_threshold=0.25, target_sizes=tgt)[0]
    return r["boxes"].cpu().numpy(), r["scores"].cpu().numpy()


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 83.3 MB/s eta 0:00:00:00:0100:01


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


preprocessor_config.json:   0%|          | 0.00/457 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.74k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.24k [00:00<?, ?B/s]

...

[transformers] You are using a model of type `sam2_video` to instantiate a model of type `sam2`. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.


model.safetensors: reconstructing file:   0%|          |  0.00B /  898MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/741 [00:00<?, ?it/s]

модели ок (SAM2)


In [ ]:
import os
from tqdm import tqdm

D = '/content/tandt/truck'
IMG_DIR  = f'{D}/images'
MASK_DIR = f'{D}/masks'
PREV_DIR = f'{D}/_mask_preview'
os.makedirs(MASK_DIR, exist_ok=True); os.makedirs(PREV_DIR, exist_ok=True)

_pp = getattr(sam_proc, "post_process_masks", None) or sam_proc.image_processor.post_process_masks

def sam_union(img, boxes):
    inp = sam_proc(img, input_boxes=[[list(map(float, b)) for b in boxes]], return_tensors="pt").to(dev)
    with torch.no_grad():
        out = sam_model(**inp, multimask_output=True)
    masks = _pp(out.pred_masks.cpu(), inp["original_sizes"].cpu())[0]
    m = masks.numpy()
    iou = out.iou_scores.cpu().numpy()[0]
    sel = np.stack([m[i, iou[i].argmax()] for i in range(m.shape[0])])
    return sel.any(0)

names = sorted(os.listdir(IMG_DIR))
no_box = []
for i, n in enumerate(tqdm(names)):
    img = Image.open(f"{IMG_DIR}/{n}").convert("RGB")
    W, H = img.size
    b, s = gdino_boxes(img, OBJECT_PROMPT, 0.3)
    if len(b) == 0:
        b, s = gdino_boxes(img, "object. vehicle. truck.", 0.22)
    keep = [bb for bb in b if 0.02 < ((bb[2]-bb[0])*(bb[3]-bb[1]))/(W*H) < 0.99]
    if not keep:
        no_box.append(n)
        mask = np.ones((H, W), bool)
    else:
        mask = sam_union(img, keep)
    Image.fromarray((mask*255).astype(np.uint8)).save(f"{MASK_DIR}/{os.path.splitext(n)[0]}.png")
print("масок:", len(os.listdir(MASK_DIR)), "| без бокса:", len(no_box))


FileNotFoundError: [Errno 2] No such file or directory: '/content/tandt/truck/images'

In [ ]:
import os

D = '/content/tandt/truck'
if not os.path.isdir(f'{D}/images'):
    os.system('wget -q --show-progress https://repo-sam.inria.fr/fungraph/3d-gaussian-splatting/datasets/input/tandt_db.zip -O /content/tandt_db.zip')
    os.system('cd /content && unzip -q -o tandt_db.zip')
print('кадров:', len(os.listdir(f'{D}/images')))


кадров: 251


In [ ]:
import os
from tqdm import tqdm

D = '/content/tandt/truck'
IMG_DIR  = f'{D}/images'
MASK_DIR = f'{D}/masks'
PREV_DIR = f'{D}/_mask_preview'
os.makedirs(MASK_DIR, exist_ok=True); os.makedirs(PREV_DIR, exist_ok=True)

_pp = getattr(sam_proc, "post_process_masks", None) or sam_proc.image_processor.post_process_masks

def sam_union(img, boxes):
    inp = sam_proc(img, input_boxes=[[list(map(float, b)) for b in boxes]], return_tensors="pt").to(dev)
    with torch.no_grad():
        out = sam_model(**inp, multimask_output=True)
    masks = _pp(out.pred_masks.cpu(), inp["original_sizes"].cpu())[0]
    m = masks.numpy()
    iou = out.iou_scores.cpu().numpy()[0]
    sel = np.stack([m[i, iou[i].argmax()] for i in range(m.shape[0])])
    return sel.any(0)

names = sorted(os.listdir(IMG_DIR))
no_box = []
for i, n in enumerate(tqdm(names)):
    img = Image.open(f"{IMG_DIR}/{n}").convert("RGB")
    W, H = img.size
    b, s = gdino_boxes(img, OBJECT_PROMPT, 0.3)
    if len(b) == 0:
        b, s = gdino_boxes(img, "object. vehicle. truck.", 0.22)
    keep = [bb for bb in b if 0.02 < ((bb[2]-bb[0])*(bb[3]-bb[1]))/(W*H) < 0.99]
    if not keep:
        no_box.append(n)
        mask = np.ones((H, W), bool)
    else:
        mask = sam_union(img, keep)
    Image.fromarray((mask*255).astype(np.uint8)).save(f"{MASK_DIR}/{os.path.splitext(n)[0]}.png")
print("масок:", len(os.listdir(MASK_DIR)), "| без бокса:", len(no_box))


100%|██████████| 251/251 [06:34<00:00,  1.57s/it]

масок: 251 | без бокса: 0


In [ ]:
!yes | TORCHDYNAMO_DISABLE=1 /content/nsenv/bin/ns-train splatfacto --data /content/tandt/truck --max-num-iterations 7000 --steps-per-save 2000 --vis tensorboard --viewer.quit-on-train-completion True --output-dir outputs_masked --pipeline.model.cull-alpha-thresh 0.1 colmap --images-path images --colmap-path sparse/0 --masks-path masks --downscale-factor 1


/bin/bash: line 1: /content/nsenv/bin/ns-train: No such file or directory


In [ ]:
import os
print('venv:', os.path.exists('/content/nsenv/bin/ns-train'))
print('датасет:', os.path.exists('/content/tandt/truck/images'))
print('маски:', len(os.listdir('/content/tandt/truck/masks')) if os.path.exists('/content/tandt/truck/masks') else 0)


venv: False
датасет: True
маски: 251


In [ ]:
%%bash
set -e
NSPY=/content/nsenv/bin/python
pip install -q uv
uv python install 3.10
uv venv --python 3.10 /content/nsenv
uv pip install --python $NSPY "torch==2.1.2" "torchvision==0.16.2" --index-url https://download.pytorch.org/whl/cu118
uv pip install --python $NSPY "setuptools<70" wheel
uv pip install --python $NSPY "gsplat==1.4.0+pt21cu118" --index-url https://docs.gsplat.studio/whl/pt21cu118 --extra-index-url https://pypi.org/simple --index-strategy unsafe-best-match
uv pip install --python $NSPY "nerfstudio==1.1.5" open3d trimesh
uv pip install --python $NSPY "numpy==1.26.4"
$NSPY -c "import torch; print('cuda', torch.cuda.is_available())"


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.1/20.1 MB 86.3 MB/s eta 0:00:00
cuda True


Installed Python 3.10.21 in 1.61s
 + cpython-3.10.21-linux-x86_64-gnu (python3.10)
Using CPython 3.10.21
Creating virtual environment at: nsenv
Activate with: source nsenv/bin/activate
Using Python 3.10.21 environment at: nsenv
Resolved 18 packages in 572ms
Prepared 18 packages in 1m 06s
Installed 18 packages in 251ms
 + certifi==2022.12.7
 + charset-normalizer==2.1.1
 + filelock==3.32.3
 + fsspec==2026.7.0
 + idna==3.4
 + jinja2==3.1.6
 + markupsafe==3.0.3
 + mpmath==1.3.0
 + networkx==3.4.2
 + numpy==2.2.6
 + pillow==12.3.0
 + requests==2.28.1
 + sympy==1.14.0
 + torch==2.1.2+cu118
 + torchvision==0.16.2+cu118
 + triton==2.1.0
 + typing-extensions==4.16.0
 + urllib3==1.26.13
Using Python 3.10.21 environment at: nsenv
Resolved 3 packages in 143ms
Prepared 3 packages in 47ms
Installed 3 packages in 6ms
 + packaging==26.3
 + setuptools==69.5.1
 + wheel==0.48.0
Using Python 3.10.21 environment at: nsenv
Resolved 19 packages in 1.68s
Prepared 8 packages in 599ms
Installed 8 packages in 57

In [ ]:
!yes | TORCHDYNAMO_DISABLE=1 /content/nsenv/bin/ns-train splatfacto --data /content/tandt/truck --max-num-iterations 7000 --steps-per-save 2000 --vis tensorboard --viewer.quit-on-train-completion True --output-dir outputs_masked --pipeline.model.cull-alpha-thresh 0.1 colmap --images-path images --colmap-path sparse/0 --masks-path masks --downscale-factor 1


[12:25:19] Using --data alias for --data.pipeline.datamanager.data                                          ]8;id=12877447;file:///content/nsenv/lib/python3.10/site-packages/nerfstudio/scripts/train.py\train.py]8;;\:]8;id=12877448;file:///content/nsenv/lib/python3.10/site-packages/nerfstudio/scripts/train.py#230\230]8;;\
──────────────────────────────────────────────────────── Config ────────────────────────────────────────────────────────
TrainerConfig(
    _target=<class 'nerfstudio.engine.trainer.Trainer'>,
    output_dir=PosixPath('outputs_masked'),
    method_name='splatfacto',
    experiment_name=None,
    project_name='nerfstudio-project',
    timestamp='2026-09-16_122519',
    machine=MachineConfig(seed=42, num_devices=1, num_machines=1, machine_rank=0, dist_url='auto', device_type='cuda'),
    logging=LoggingConfig(
        relative_log_dir=PosixPath('.'),
        steps_per_log=10,
        max_buffer_size=20,
        local_writer=LocalWriterConfig(
            _target=

In [ ]:
import subprocess

code = r'''
import glob, shutil
from PIL import Image
from nerfstudio.data.utils.colmap_parsing_utils import read_cameras_binary, write_cameras_binary, Camera
sp, im = "/content/tandt/truck/sparse/0", "/content/tandt/truck/images"
cams = read_cameras_binary(f"{sp}/cameras.bin")
w, h = Image.open(sorted(glob.glob(im + "/*"))[0]).size
new = {}
for k, c in cams.items():
    sx, sy = w / c.width, h / c.height
    p = list(c.params)
    if c.model == "PINHOLE":
        p = [p[0]*sx, p[1]*sy, p[2]*sx, p[3]*sy]
    new[k] = Camera(id=c.id, model=c.model, width=w, height=h, params=p)
shutil.copy(f"{sp}/cameras.bin", f"{sp}/cameras.bin.orig")
write_cameras_binary(new, f"{sp}/cameras.bin")
print("готово:", [v.width for v in read_cameras_binary(f"{sp}/cameras.bin").values()][:1])
'''
r = subprocess.run(['/content/nsenv/bin/python', '-c', code], capture_output=True, text=True)
print(r.stdout or r.stderr[-2000:])

FileNotFoundError: [Errno 2] No such file or directory: '/content/nsenv/bin/python'

In [ ]:
import os
print('venv:', os.path.exists('/content/nsenv/bin/ns-train'))
print('датасет:', os.path.exists('/content/tandt/truck/images'))
print('маски:', len(os.listdir('/content/tandt/truck/masks')) if os.path.exists('/content/tandt/truck/masks') else 0)


venv: False
датасет: False
маски: 0


In [ ]:
%%bash
set -e
NSPY=/content/nsenv/bin/python

if [ ! -x /content/nsenv/bin/ns-train ]; then
  pip install -q uv
  uv python install 3.10
  uv venv --python 3.10 /content/nsenv
  uv pip install --python $NSPY "torch==2.1.2" "torchvision==0.16.2" --index-url https://download.pytorch.org/whl/cu118
  uv pip install --python $NSPY "setuptools<70" wheel
  uv pip install --python $NSPY "gsplat==1.4.0+pt21cu118" --index-url https://docs.gsplat.studio/whl/pt21cu118 --extra-index-url https://pypi.org/simple --index-strategy unsafe-best-match
  uv pip install --python $NSPY "nerfstudio==1.1.5" open3d trimesh
  uv pip install --python $NSPY "numpy==1.26.4"
fi

if [ ! -d /content/tandt/truck/images ]; then
  wget -q https://repo-sam.inria.fr/fungraph/3d-gaussian-splatting/datasets/input/tandt_db.zip -O /content/tandt_db.zip
  cd /content && unzip -q -o tandt_db.zip
fi

if [ ! -f /content/tandt/truck/sparse/0/cameras.bin.orig ]; then
$NSPY -c "
import glob, shutil
from PIL import Image
from nerfstudio.data.utils.colmap_parsing_utils import read_cameras_binary, write_cameras_binary, Camera
sp, im = '/content/tandt/truck/sparse/0', '/content/tandt/truck/images'
cams = read_cameras_binary(f'{sp}/cameras.bin')
w, h = Image.open(sorted(glob.glob(im + '/*'))[0]).size
new = {}
for k, c in cams.items():
    sx, sy = w / c.width, h / c.height
    p = list(c.params)
    if c.model == 'PINHOLE':
        p = [p[0]*sx, p[1]*sy, p[2]*sx, p[3]*sy]
    new[k] = Camera(id=c.id, model=c.model, width=w, height=h, params=p)
shutil.copy(f'{sp}/cameras.bin', f'{sp}/cameras.bin.orig')
write_cameras_binary(new, f'{sp}/cameras.bin')
"
fi

echo '=== готово ==='
$NSPY -c "import torch; print('cuda', torch.cuda.is_available())"
ls /content/tandt/truck/images | wc -l


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.1/20.1 MB 98.5 MB/s eta 0:00:00
=== готово ===
cuda True
251


Installed Python 3.10.21 in 1.52s
 + cpython-3.10.21-linux-x86_64-gnu (python3.10)
Using CPython 3.10.21
Creating virtual environment at: nsenv
Activate with: source nsenv/bin/activate
Using Python 3.10.21 environment at: nsenv
Resolved 18 packages in 1.02s
Prepared 18 packages in 47.43s
Installed 18 packages in 271ms
 + certifi==2022.12.7
 + charset-normalizer==2.1.1
 + filelock==3.32.3
 + fsspec==2026.7.0
 + idna==3.4
 + jinja2==3.1.6
 + markupsafe==3.0.3
 + mpmath==1.3.0
 + networkx==3.4.2
 + numpy==2.2.6
 + pillow==12.3.0
 + requests==2.28.1
 + sympy==1.14.0
 + torch==2.1.2+cu118
 + torchvision==0.16.2+cu118
 + triton==2.1.0
 + typing-extensions==4.16.0
 + urllib3==1.26.13
Using Python 3.10.21 environment at: nsenv
Resolved 3 packages in 147ms
Prepared 3 packages in 53ms
Installed 3 packages in 6ms
 + packaging==26.3
 + setuptools==69.5.1
 + wheel==0.48.0
Using Python 3.10.21 environment at: nsenv
Resolved 19 packages in 1.27s
Prepared 8 packages in 666ms
Installed 8 packages in 70

In [ ]:
!pip install -q -U transformers
import torch, numpy as np
from PIL import Image
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection, Sam2Model, Sam2Processor

dev = "cuda"
OBJECT_PROMPT = "a truck."
GD, SAM = "IDEA-Research/grounding-dino-base", "facebook/sam2-hiera-large"
gd_proc = AutoProcessor.from_pretrained(GD)
gd_model = AutoModelForZeroShotObjectDetection.from_pretrained(GD).to(dev).eval()
sam_proc = Sam2Processor.from_pretrained(SAM)
sam_model = Sam2Model.from_pretrained(SAM).to(dev).eval()
print("модели ок (SAM2)")

def gdino_boxes(img, text=OBJECT_PROMPT, thr=0.3):
    inp = gd_proc(images=img, text=text, return_tensors="pt").to(dev)
    with torch.no_grad():
        out = gd_model(**inp)
    tgt = [img.size[::-1]]
    try:
        r = gd_proc.post_process_grounded_object_detection(out, inp.input_ids, threshold=thr, text_threshold=0.25, target_sizes=tgt)[0]
    except TypeError:
        r = gd_proc.post_process_grounded_object_detection(out, inp.input_ids, box_threshold=thr, text_threshold=0.25, target_sizes=tgt)[0]
    return r["boxes"].cpu().numpy(), r["scores"].cpu().numpy()


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 106.2 MB/s eta 0:00:0000:010:01


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


preprocessor_config.json:   0%|          | 0.00/457 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.74k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.24k [00:00<?, ?B/s]

...

[transformers] You are using a model of type `sam2_video` to instantiate a model of type `sam2`. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.


model.safetensors: reconstructing file:   0%|          |  0.00B /  898MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/741 [00:00<?, ?it/s]

модели ок (SAM2)


In [ ]:
import os
from tqdm import tqdm

D = '/content/tandt/truck'
IMG_DIR  = f'{D}/images'
MASK_DIR = f'{D}/masks'
os.makedirs(MASK_DIR, exist_ok=True)

_pp = getattr(sam_proc, "post_process_masks", None) or sam_proc.image_processor.post_process_masks

def sam_union(img, boxes):
    inp = sam_proc(img, input_boxes=[[list(map(float, b)) for b in boxes]], return_tensors="pt").to(dev)
    with torch.no_grad():
        out = sam_model(**inp, multimask_output=True)
    masks = _pp(out.pred_masks.cpu(), inp["original_sizes"].cpu())[0]
    m = masks.numpy()
    iou = out.iou_scores.cpu().numpy()[0]
    sel = np.stack([m[i, iou[i].argmax()] for i in range(m.shape[0])])
    return sel.any(0)

names = sorted(os.listdir(IMG_DIR))
no_box = []
for i, n in enumerate(tqdm(names)):
    img = Image.open(f"{IMG_DIR}/{n}").convert("RGB")
    W, H = img.size
    b, s = gdino_boxes(img, OBJECT_PROMPT, 0.3)
    if len(b) == 0:
        b, s = gdino_boxes(img, "object. vehicle. truck.", 0.22)
    keep = [bb for bb in b if 0.02 < ((bb[2]-bb[0])*(bb[3]-bb[1]))/(W*H) < 0.99]
    if not keep:
        no_box.append(n)
        mask = np.ones((H, W), bool)
    else:
        mask = sam_union(img, keep)
    Image.fromarray((mask*255).astype(np.uint8)).save(f"{MASK_DIR}/{os.path.splitext(n)[0]}.png")
print("масок:", len(os.listdir(MASK_DIR)), "| без бокса:", len(no_box))


100%|██████████| 251/251 [06:18<00:00,  1.51s/it]

масок: 251 | без бокса: 0


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, time
os.makedirs('/content/drive/MyDrive/nerf_cache', exist_ok=True)
CACHE = '/content/drive/MyDrive/nerf_cache/masks_251.tar.gz'
os.system(f'tar -czf {CACHE} -C /content/tandt/truck masks')
time.sleep(3)
print('размер на Drive:', os.path.getsize(CACHE), 'байт')
print(os.listdir('/content/drive/MyDrive/nerf_cache'))


Mounted at /content/drive
размер на Drive: 767443 байт
['masks_251.tar.gz']


In [ ]:
!yes | TORCHDYNAMO_DISABLE=1 /content/nsenv/bin/ns-train splatfacto --data /content/tandt/truck --max-num-iterations 7000 --steps-per-save 2000 --vis tensorboard --viewer.quit-on-train-completion True --output-dir outputs_masked --pipeline.model.cull-alpha-thresh 0.1 colmap --images-path images --colmap-path sparse/0 --masks-path masks --downscale-factor 1


[13:02:37] Using --data alias for --data.pipeline.datamanager.data                                          ]8;id=6329751;file:///content/nsenv/lib/python3.10/site-packages/nerfstudio/scripts/train.py\train.py]8;;\:]8;id=6329752;file:///content/nsenv/lib/python3.10/site-packages/nerfstudio/scripts/train.py#230\230]8;;\
──────────────────────────────────────────────────────── Config ────────────────────────────────────────────────────────
TrainerConfig(
    _target=<class 'nerfstudio.engine.trainer.Trainer'>,
    output_dir=PosixPath('outputs_masked'),
    method_name='splatfacto',
    experiment_name=None,
    project_name='nerfstudio-project',
    timestamp='2026-09-17_130237',
    machine=MachineConfig(seed=42, num_devices=1, num_machines=1, machine_rank=0, dist_url='auto', device_type='cuda'),
    logging=LoggingConfig(
        relative_log_dir=PosixPath('.'),
        steps_per_log=10,
        max_buffer_size=20,
        local_writer=LocalWriterConfig(
            _target=<c

In [ ]:
!ls outputs_masked/truck/splatfacto/*/config.yml


outputs_masked/truck/splatfacto/2026-09-17_130237/config.yml


In [ ]:
!/content/nsenv/bin/ns-export gaussian-splat --load-config outputs_masked/truck/splatfacto/2026-09-17_130237/config.yml --output-dir exports/truck_masked
!ls -la exports/truck_masked


Loading latest checkpoint from load_dir
✅ Done loading checkpoint from outputs_masked/truck/splatfacto/2026-09-17_130237/nerfstudio_models/step-000006999.ckpt
0 Gaussians have NaN/Inf and 2 have low opacity, only export 120246/120248
total 29132
drwxr-xr-x 2 root root     4096 Sep 17 13:08 .
drwxr-xr-x 3 root root     4096 Sep 17 13:08 ..
-rw-r--r-- 1 root root 29822601 Sep 17 13:08 splat.ply


In [ ]:
import shutil, os
shutil.copy('exports/truck_masked/splat.ply', '/content/drive/MyDrive/nerf_cache/splat_251_ref.ply')
print(os.path.getsize('/content/drive/MyDrive/nerf_cache/splat_251_ref.ply'))


29822601


In [ ]:
script = r'''
import glob
import numpy as np
import open3d as o3d
import trimesh

ply = "exports/truck_masked/splat.ply"
pcd = o3d.io.read_point_cloud(ply)
print("Punkte:", len(pcd.points))
pcd, _ = pcd.remove_statistical_outlier(nb_neighbors=20, std_ratio=2.0)
print("nach Bereinigung:", len(pcd.points))

nn = o3d.geometry.KDTreeFlann(pcd)
step = max(1, len(pcd.points) // 2000)
d = []
for i in range(0, len(pcd.points), step):
    _, idx, dist2 = nn.search_knn_vector_3d(pcd.points[i], 2)
    d.append(np.sqrt(dist2[1]))
eps = 5 * float(np.median(d))
labels = np.array(pcd.cluster_dbscan(eps=eps, min_points=10))
if labels.max() >= 0:
    biggest = np.bincount(labels[labels >= 0]).argmax()
    pcd = pcd.select_by_index(np.where(labels == biggest)[0])
print("nach DBSCAN:", len(pcd.points))

pcd.estimate_normals(o3d.geometry.KDTreeSearchParamHybrid(radius=0.1, max_nn=30))
pcd.orient_normals_consistent_tangent_plane(30)
mesh, dens = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(pcd, depth=10)
dens = np.asarray(dens)
mesh.remove_vertices_by_mask(dens <= np.quantile(dens, 0.1))
mesh.remove_degenerate_triangles(); mesh.remove_duplicated_vertices()
mesh.remove_duplicated_triangles(); mesh.remove_non_manifold_edges()
out = "exports/truck_masked/truck_object_ref.stl"
o3d.io.write_triangle_mesh(out, mesh)

m = trimesh.load(out)
m.fill_holes(); m.fix_normals(); m.process(validate=True)
m.export(out)
print("STL:", out, "| Ecken:", len(m.vertices), "| Dreiecke:", len(m.faces))
'''
open('/content/mesh_ref.py', 'w').write(script)
import subprocess
r = subprocess.run(['/content/nsenv/bin/python', '/content/mesh_ref.py'], capture_output=True, text=True)
print(r.stdout)
if r.returncode:
    print('STDERR:\n', r.stderr[-3000:])


Punkte: 120246
nach Bereinigung: 117807
nach DBSCAN: 77643
[Open3D WARNING] Write STL failed: compute normals first.

STDERR:
 Traceback (most recent call last):
  File "/content/mesh_ref.py", line 37, in <module>
    m.fill_holes(); m.fix_normals(); m.process(validate=True)
AttributeError: 'Scene' object has no attribute 'fill_holes'



In [ ]:
script = r'''
import glob
import numpy as np
import open3d as o3d
import trimesh

ply = "exports/truck_masked/splat.ply"
pcd = o3d.io.read_point_cloud(ply)
print("Punkte:", len(pcd.points))
pcd, _ = pcd.remove_statistical_outlier(nb_neighbors=20, std_ratio=2.0)
print("nach Bereinigung:", len(pcd.points))

nn = o3d.geometry.KDTreeFlann(pcd)
step = max(1, len(pcd.points) // 2000)
d = []
for i in range(0, len(pcd.points), step):
    _, idx, dist2 = nn.search_knn_vector_3d(pcd.points[i], 2)
    d.append(np.sqrt(dist2[1]))
eps = 5 * float(np.median(d))
labels = np.array(pcd.cluster_dbscan(eps=eps, min_points=10))
if labels.max() >= 0:
    biggest = np.bincount(labels[labels >= 0]).argmax()
    pcd = pcd.select_by_index(np.where(labels == biggest)[0])
print("nach DBSCAN:", len(pcd.points))

pcd.estimate_normals(o3d.geometry.KDTreeSearchParamHybrid(radius=0.1, max_nn=30))
pcd.orient_normals_consistent_tangent_plane(30)
mesh, dens = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(pcd, depth=10)
dens = np.asarray(dens)
mesh.remove_vertices_by_mask(dens <= np.quantile(dens, 0.1))
mesh.remove_degenerate_triangles(); mesh.remove_duplicated_vertices()
mesh.remove_duplicated_triangles(); mesh.remove_non_manifold_edges()
mesh.compute_triangle_normals()
out = "exports/truck_masked/truck_object_ref.stl"
o3d.io.write_triangle_mesh(out, mesh)

m = trimesh.load(out)
m.fill_holes(); m.fix_normals(); m.process(validate=True)
m.export(out)
print("STL:", out, "| Ecken:", len(m.vertices), "| Dreiecke:", len(m.faces))
'''
open('/content/mesh_ref.py', 'w').write(script)
import subprocess
r = subprocess.run(['/content/nsenv/bin/python', '/content/mesh_ref.py'], capture_output=True, text=True)
print(r.stdout)
if r.returncode:
    print('STDERR:\n', r.stderr[-3000:])


Punkte: 120246
nach Bereinigung: 117807
nach DBSCAN: 77643
STL: exports/truck_masked/truck_object_ref.stl | Ecken: 222017 | Dreiecke: 430796



In [ ]:
import shutil
shutil.copy('exports/truck_masked/truck_object_ref.stl', '/content/drive/MyDrive/nerf_cache/truck_object_ref.stl')
print('сохранено')


сохранено


In [ ]:
import os
print('sparse20:', os.path.exists('/content/sparse_test/processed/transforms.json'))
print('sparse30:', os.path.exists('/content/sparse_test30/processed/transforms.json'))


sparse20: False
sparse30: False


In [ ]:
import os
print('colmap:', os.system('which colmap') == 0)


colmap: False


In [ ]:
!apt-get -qq update
!apt-get -qq install -y colmap
!colmap -h | head -2


W: https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt/trusted.gpg), see the DEPRECATION section in apt-key(8) for details.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Extracting templates from packages: 100%
Selecting previously unselected package libevdev2:amd64.
(Reading database ... 126952 files and directories currently installed.)
Preparing to unpack .../00-libevdev2_1.13.1+dfsg-1build1_amd64.deb ...
Unpacking libevdev2:amd64 (1.13.1+dfsg-1build1) ...
Selecting previously unselected package at-spi2-common.
Preparing to unpack .../01-at-spi2-common_2.52.0-1build1_all.deb ...
Unpacking at-spi2-common (2.52.0-1build1) ...
Selecting previously unselected package libatspi2.0-0t64:amd64.
Preparing to unpack .../02-libatspi2.0-0t64_2.52.0-1build1_amd64.deb .

In [ ]:
import os, shutil, random

SRC = '/content/tandt/truck/images'
names = sorted(os.listdir(SRC))
random.seed(0)

for n_photos, tag in [(20, 'sparse_test'), (30, 'sparse_test30')]:
    RAW = f'/content/{tag}/raw'
    os.makedirs(RAW, exist_ok=True)
    subset = sorted(random.Random(0).sample(names, n_photos))
    for n in subset:
        shutil.copy(f'{SRC}/{n}', f'{RAW}/{n}')
    print(tag, '- скопировано:', len(os.listdir(RAW)))


sparse_test - скопировано: 20
sparse_test30 - скопировано: 30


In [ ]:
!rm -rf /content/sparse_test/processed
!QT_QPA_PLATFORM=offscreen /content/nsenv/bin/ns-process-data images \
  --data /content/sparse_test/raw \
  --output-dir /content/sparse_test/processed \
  --matching-method exhaustive \
  --no-gpu


[13:21:02] 🎉 Done copying images with prefix 'frame_'.                                        ]8;id=7279142;file:///content/nsenv/lib/python3.10/site-packages/nerfstudio/process_data/process_data_utils.py\process_data_utils.py]8;;\:]8;id=7279143;file:///content/nsenv/lib/python3.10/site-packages/nerfstudio/process_data/process_data_utils.py#348\348]8;;\
(  ●   ) Copying images...
🌘  Running COLMAP feature extractor...tor...
[13:21:31] 🎉 Done extracting COLMAP features.                                                       ]8;id=7279150;file:///content/nsenv/lib/python3.10/site-packages/nerfstudio/process_data/colmap_utils.py\colmap_utils.py]8;;\:]8;id=7279151;file:///content/nsenv/lib/python3.10/site-packages/nerfstudio/process_data/colmap_utils.py#137\137]8;;\
🏃  Running COLMAP feature matcher...0m
[13:23:13] 🎉 Done matching COLMAP features.                                                         ]8;id=7279157;file:///content/nsenv/lib/python3.10/site-packages/nerfstu

In [ ]:
!rm -rf /content/sparse_test30/processed
!QT_QPA_PLATFORM=offscreen /content/nsenv/bin/ns-process-data images \
  --data /content/sparse_test30/raw \
  --output-dir /content/sparse_test30/processed \
  --matching-method exhaustive \
  --no-gpu


[13:25:26] 🎉 Done copying images with prefix 'frame_'.                                        ]8;id=14662674;file:///content/nsenv/lib/python3.10/site-packages/nerfstudio/process_data/process_data_utils.py\process_data_utils.py]8;;\:]8;id=14662675;file:///content/nsenv/lib/python3.10/site-packages/nerfstudio/process_data/process_data_utils.py#348\348]8;;\
(  ●   ) Copying images...
🌑  Running COLMAP feature extractor...tor...
[13:26:07] 🎉 Done extracting COLMAP features.                                                       ]8;id=14662682;file:///content/nsenv/lib/python3.10/site-packages/nerfstudio/process_data/colmap_utils.py\colmap_utils.py]8;;\:]8;id=14662683;file:///content/nsenv/lib/python3.10/site-packages/nerfstudio/process_data/colmap_utils.py#137\137]8;;\
🚶  Running COLMAP feature matcher...0m
[13:29:46] 🎉 Done matching COLMAP features.                                                         ]8;id=14662689;file:///content/nsenv/lib/python3.10/site-packages/ne

In [ ]:
import os
os.system('tar -czf /content/drive/MyDrive/nerf_cache/sparse30_processed.tar.gz -C /content/sparse_test30 processed')
print(os.path.getsize('/content/drive/MyDrive/nerf_cache/sparse30_processed.tar.gz'))


25404506


In [ ]:
import os
from tqdm import tqdm

IMG_DIR30 = '/content/sparse_test30/processed/images'
MASK_DIR30 = '/content/sparse_test30/processed/masks'
os.makedirs(MASK_DIR30, exist_ok=True)

names30 = sorted(os.listdir(IMG_DIR30))
print('фото в processed:', len(names30))

no_box30 = []
for n in tqdm(names30):
    img = Image.open(f"{IMG_DIR30}/{n}").convert("RGB")
    W, H = img.size
    b, s = gdino_boxes(img, OBJECT_PROMPT, 0.3)
    if len(b) == 0:
        b, s = gdino_boxes(img, "object. vehicle. truck.", 0.22)
    keep = [bb for bb in b if 0.02 < ((bb[2]-bb[0])*(bb[3]-bb[1]))/(W*H) < 0.99]
    if not keep:
        no_box30.append(n)
        mask = np.ones((H, W), bool)
    else:
        mask = sam_union(img, keep)
    Image.fromarray((mask*255).astype(np.uint8)).save(f"{MASK_DIR30}/{os.path.splitext(n)[0]}.png")
print("масок:", len(os.listdir(MASK_DIR30)), "| без бокса:", len(no_box30))


фото в processed: 30


100%|██████████| 30/30 [00:44<00:00,  1.49s/it]

масок: 30 | без бокса: 0


In [ ]:
!yes | TORCHDYNAMO_DISABLE=1 /content/nsenv/bin/ns-train splatfacto \
  --data /content/sparse_test30/processed \
  --max-num-iterations 3000 --steps-per-save 1000 \
  --vis tensorboard --viewer.quit-on-train-completion True \
  --output-dir outputs_sparse30 --pipeline.model.cull-alpha-thresh 0.1 \
  nerfstudio-data --masks-path masks


╭─ Unrecognized options ───────────────╮
│ Unrecognized or misplaced options:   │
│   --masks-path (applied to           │
│ /content/nsenv/bin/ns-train          │
│ splatfacto nerfstudio-data)          │
│   masks (applied to                  │
│ /content/nsenv/bin/ns-train          │
│ splatfacto nerfstudio-data)          │
│                                      │
│ Arguments are applied to the         │
│ directly preceding subcommand, so    │
│ ordering can matter.                 │
│ ──────────────────────────────────── │
│ Perhaps you meant:                   │
│     --masks-path {None}|PATH         │
│         Path to masks directory. If  │
│         not set, masks are not       │
│         loaded. (default: None)      │
│         in                           │
│         /content/nsenv/bin/ns-train  │
│         splatfacto colmap --help     │
│         in                           │
│         /content/nsenv/bin/ns-train  │
│         splatfacto colmap --help     │
│         in    

In [ ]:
!find /content/sparse_test30/processed -maxdepth 3 -type d


/content/sparse_test30/processed
/content/sparse_test30/processed/images
/content/sparse_test30/processed/masks
/content/sparse_test30/processed/colmap
/content/sparse_test30/processed/colmap/sparse
/content/sparse_test30/processed/colmap/sparse/0
/content/sparse_test30/processed/images_2
/content/sparse_test30/processed/images_8
/content/sparse_test30/processed/images_4


In [ ]:
!yes | TORCHDYNAMO_DISABLE=1 /content/nsenv/bin/ns-train splatfacto \
  --data /content/sparse_test30/processed \
  --max-num-iterations 3000 --steps-per-save 1000 \
  --vis tensorboard --viewer.quit-on-train-completion True \
  --output-dir outputs_sparse30 --pipeline.model.cull-alpha-thresh 0.1 \
  colmap --images-path images --colmap-path colmap/sparse/0 --masks-path masks --downscale-factor 1


[13:39:34] Using --data alias for --data.pipeline.datamanager.data                                          ]8;id=15420382;file:///content/nsenv/lib/python3.10/site-packages/nerfstudio/scripts/train.py\train.py]8;;\:]8;id=15420383;file:///content/nsenv/lib/python3.10/site-packages/nerfstudio/scripts/train.py#230\230]8;;\
──────────────────────────────────────────────────────── Config ────────────────────────────────────────────────────────
TrainerConfig(
    _target=<class 'nerfstudio.engine.trainer.Trainer'>,
    output_dir=PosixPath('outputs_sparse30'),
    method_name='splatfacto',
    experiment_name=None,
    project_name='nerfstudio-project',
    timestamp='2026-09-17_133934',
    machine=MachineConfig(seed=42, num_devices=1, num_machines=1, machine_rank=0, dist_url='auto', device_type='cuda'),
    logging=LoggingConfig(
        relative_log_dir=PosixPath('.'),
        steps_per_log=10,
        max_buffer_size=20,
        local_writer=LocalWriterConfig(
            _targe

In [ ]:
!ls outputs_sparse30/images/splatfacto/*/config.yml 2>/dev/null || ls outputs_sparse30/*/splatfacto/*/config.yml


outputs_sparse30/processed/splatfacto/2026-09-17_133934/config.yml


In [ ]:
!/content/nsenv/bin/ns-export gaussian-splat --load-config outputs_sparse30/processed/splatfacto/2026-09-17_133934/config.yml --output-dir exports/sparse30
!ls -la exports/sparse30


Loading latest checkpoint from load_dir
✅ Done loading checkpoint from 
outputs_sparse30/processed/splatfacto/2026-09-17_133934/nerfstudio_models/step-000002999.ckpt
0 Gaussians have NaN/Inf and 2 have low opacity, only export 75534/75536
total 18304
drwxr-xr-x 2 root root     4096 Sep 17 13:58 .
drwxr-xr-x 4 root root     4096 Sep 17 13:58 ..
-rw-r--r-- 1 root root 18734024 Sep 17 13:58 splat.ply


In [ ]:
import shutil
shutil.copy('exports/sparse30/splat.ply', '/content/drive/MyDrive/nerf_cache/splat_sparse30.ply')
print('сохранено')


сохранено


In [ ]:
script = r'''
import numpy as np
import open3d as o3d
import trimesh

ply = "exports/sparse30/splat.ply"
pcd = o3d.io.read_point_cloud(ply)
print("Punkte:", len(pcd.points))
pcd, _ = pcd.remove_statistical_outlier(nb_neighbors=20, std_ratio=2.0)
print("nach Bereinigung:", len(pcd.points))

nn = o3d.geometry.KDTreeFlann(pcd)
step = max(1, len(pcd.points) // 2000)
d = []
for i in range(0, len(pcd.points), step):
    _, idx, dist2 = nn.search_knn_vector_3d(pcd.points[i], 2)
    d.append(np.sqrt(dist2[1]))
eps = 5 * float(np.median(d))
labels = np.array(pcd.cluster_dbscan(eps=eps, min_points=10))
if labels.max() >= 0:
    biggest = np.bincount(labels[labels >= 0]).argmax()
    pcd = pcd.select_by_index(np.where(labels == biggest)[0])
print("nach DBSCAN:", len(pcd.points))

pcd.estimate_normals(o3d.geometry.KDTreeSearchParamHybrid(radius=0.1, max_nn=30))
pcd.orient_normals_consistent_tangent_plane(30)
mesh, dens = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(pcd, depth=9)
dens = np.asarray(dens)
mesh.remove_vertices_by_mask(dens <= np.quantile(dens, 0.1))
mesh.remove_degenerate_triangles(); mesh.remove_duplicated_vertices()
mesh.remove_duplicated_triangles(); mesh.remove_non_manifold_edges()
mesh.compute_triangle_normals()
out = "exports/sparse30/sparse30_object.stl"
o3d.io.write_triangle_mesh(out, mesh)

m = trimesh.load(out)
m.fill_holes(); m.fix_normals(); m.process(validate=True)
m.export(out)
print("STL:", out, "| Ecken:", len(m.vertices), "| Dreiecke:", len(m.faces))
'''
open('/content/mesh_sparse30.py', 'w').write(script)
import subprocess
r = subprocess.run(['/content/nsenv/bin/python', '/content/mesh_sparse30.py'], capture_output=True, text=True)
print(r.stdout)
if r.returncode:
    print('STDERR:\n', r.stderr[-3000:])


Punkte: 75534
nach Bereinigung: 72815
nach DBSCAN: 38644
STL: exports/sparse30/sparse30_object.stl | Ecken: 126168 | Dreiecke: 245182



In [ ]:
script = r'''
import numpy as np
import open3d as o3d
import trimesh

def load_pts(path, n=30000):
    m = trimesh.load(path)
    pts, _ = trimesh.sample.sample_surface(m, n)
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(pts)
    return pcd

def normalize(pcd):
    pts = np.asarray(pcd.points)
    c = pts.mean(0)
    pts = pts - c
    scale = np.linalg.norm(pts, axis=1).max()
    pts = pts / scale
    out = o3d.geometry.PointCloud()
    out.points = o3d.utility.Vector3dVector(pts)
    return out

ref = normalize(load_pts("exports/truck_masked/truck_object_ref.stl"))
sparse = normalize(load_pts("exports/sparse30/sparse30_object.stl"))

# перебираем повороты вокруг вертикальной оси (Z), т.к. масштаб/сдвиг уже сняты нормализацией,
# но угол разворота между двумя независимыми COLMAP-реконструкциями произвольный
best = None
for deg in range(0, 360, 15):
    R = o3d.geometry.get_rotation_matrix_from_axis_angle([0, 0, np.deg2rad(deg)])
    cand = o3d.geometry.PointCloud(sparse)
    cand.rotate(R, center=(0, 0, 0))
    reg = o3d.pipelines.registration.registration_icp(
        cand, ref, 0.05, np.eye(4),
        o3d.pipelines.registration.TransformationEstimationPointToPoint()
    )
    if best is None or reg.inlier_rmse < best[0]:
        best = (reg.inlier_rmse, deg, reg.transformation, R)

rmse, deg, T, R0 = best
print("лучший начальный угол:", deg, "| rmse после ICP:", rmse)

sparse_aligned = o3d.geometry.PointCloud(sparse)
sparse_aligned.rotate(R0, center=(0, 0, 0))
sparse_aligned.transform(T)

# финальный дожим ICP
reg2 = o3d.pipelines.registration.registration_icp(
    sparse_aligned, ref, 0.03, np.eye(4),
    o3d.pipelines.registration.TransformationEstimationPointToPlane()
    if False else o3d.pipelines.registration.TransformationEstimationPointToPoint()
)
sparse_aligned.transform(reg2.transformation)

d1 = np.asarray(sparse_aligned.compute_point_cloud_distance(ref))
d2 = np.asarray(ref.compute_point_cloud_distance(sparse_aligned))
chamfer = d1.mean() + d2.mean()
print(f"Chamfer distance (нормализовано на размер объекта): {chamfer:.4f}")
print(f"  -> sparse30 к ref: {d1.mean():.4f} (медиана {np.median(d1):.4f})")
print(f"  -> ref к sparse30: {d2.mean():.4f} (медиана {np.median(d2):.4f})")
'''
open('/content/chamfer.py', 'w').write(script)
import subprocess
r = subprocess.run(['/content/nsenv/bin/python', '/content/chamfer.py'], capture_output=True, text=True)
print(r.stdout)
if r.returncode:
    print('STDERR:\n', r.stderr[-3000:])


лучший начальный угол: 150 | rmse после ICP: 0.0262848505761137
Chamfer distance (нормализовано на размер объекта): 0.2497
  -> sparse30 к ref: 0.1189 (медиана 0.0589)
  -> ref к sparse30: 0.1308 (медиана 0.0614)



## 9. Реальный фототест: Amazon Berkeley Objects (ABO) — настоящие фото + внешняя CAD-эталонная модель

Всё выше (Teil 1-8, sparse-view тесты) использует только датасет `truck` (Tanks & Temples) —
не настоящие фото пользователя. Своих фото объекта сделать негде, поэтому вместо них
используется публичный датасет **Amazon Berkeley Objects (ABO)** (CC BY 4.0, (c) Amazon.com,
UC Berkeley — https://amazon-berkeley-objects.s3.amazonaws.com/index.html): это НАСТОЯЩИЕ
фотографии (не рендеры) реальных физических товаров на поворотном столе (72 кадра, шаг 5°),
и для части товаров дополнительно есть **настоящая CAD-модель** (glTF) того же физического
объекта — то есть впервые можно посчитать Chamfer-дистанцию против настоящей внешней
эталонной геометрии, а не self-consistency proxy из раздела 5 PROJECT_STATUS.md.

Объект: кресло "Ravenna Home Radford Modern Curved" (item_id `B07F2X8K62`),
spin_id `f8022078`, 72 фото + CAD (glb). Прогоняем через весь pipeline (COLMAP с нуля →
Grounding DINO + SAM2 маски → masked splatfacto → Poisson → STL) на трёх уровнях: dense (72,
референс), N=30, N=20 — и сравниваем каждый результат Chamfer-дистанцией с настоящим CAD.

**Обязательно указывать атрибуцию при публикации/сдаче:** данные — (c) Amazon.com, датасет —
Amazon Berkeley Objects, лицензия CC BY 4.0.

In [ ]:
import os, subprocess

ABO_ITEM = "B07F2X8K62"
ABO_SPIN_ID = "f8022078"
ABO_SPIN_PREFIX = ABO_SPIN_ID[:2]
ABO_N_FRAMES = 72
D_ABO = "/content/abo_chair"
os.makedirs(f"{D_ABO}/raw", exist_ok=True)

base = "https://amazon-berkeley-objects.s3.amazonaws.com"
for az in range(ABO_N_FRAMES):
    fn = f"{ABO_SPIN_ID}_{az:02d}.jpg"
    url = f"{base}/spins/original/{ABO_SPIN_PREFIX}/{ABO_SPIN_ID}/{fn}"
    out = f"{D_ABO}/raw/{fn}"
    if not os.path.exists(out):
        subprocess.run(["curl", "-s", "-L", url, "-o", out], check=True)
print("фото скачано:", len(os.listdir(f"{D_ABO}/raw")))

cad_dir = f"{D_ABO}/cad"
os.makedirs(cad_dir, exist_ok=True)
cad_url = f"{base}/3dmodels/original/{ABO_ITEM[-1]}/{ABO_ITEM}.glb"
subprocess.run(["curl", "-s", "-L", cad_url, "-o", f"{cad_dir}/{ABO_ITEM}.glb"], check=True)
print("CAD скачан:", os.listdir(cad_dir))
print("Источник: Amazon Berkeley Objects, CC BY 4.0, (c) Amazon.com")

In [ ]:
import shutil
import numpy as np

def make_subset(n, raw_dir=f"{D_ABO}/raw", out_name=None):
    out_name = out_name or f"raw_N{n}"
    out_dir = f"{D_ABO}/{out_name}"
    if os.path.isdir(out_dir):
        shutil.rmtree(out_dir)
    os.makedirs(out_dir)
    names = sorted(os.listdir(raw_dir))
    idx = sorted(set(np.linspace(0, len(names) - 1, n).round().astype(int).tolist()))
    for i in idx:
        shutil.copy(f"{raw_dir}/{names[i]}", out_dir)
    print(out_name, "->", len(os.listdir(out_dir)), "фото")

make_subset(ABO_N_FRAMES, out_name="raw_dense")  # полная turntable-серия как референс
make_subset(30)
make_subset(20)

In [ ]:
def run_colmap(subset_dir_name):
    raw = f"{D_ABO}/{subset_dir_name}"
    processed = f"{D_ABO}/{subset_dir_name}_processed"
    subprocess.run(f"rm -rf {processed}", shell=True)
    cmd = (
        f"QT_QPA_PLATFORM=offscreen /content/nsenv/bin/ns-process-data images "
        f"--data {raw} --output-dir {processed} --matching-method exhaustive --no-gpu"
    )
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(subset_dir_name, "-> returncode", r.returncode)
    print(r.stdout[-1500:])
    if r.returncode:
        print(r.stderr[-1500:])

for name in ["raw_dense", "raw_N30", "raw_N20"]:
    run_colmap(name)

In [ ]:
import json as _json

for name in ["raw_dense", "raw_N30", "raw_N20"]:
    tf = f"{D_ABO}/{name}_processed/transforms.json"
    if os.path.exists(tf):
        n = len(_json.load(open(tf))["frames"])
        print(name, "-> зарегистрировано кадров:", n)
    else:
        print(name, "-> transforms.json нет (COLMAP не смог зарегистрировать ни одного кадра)")

In [ ]:
# Предполагается, что ячейки из Teil 4 (gdino_boxes, sam_union, модели GD/SAM2) уже выполнены
OBJECT_PROMPT_ABO = "a chair."

def make_masks(processed_dir, prompt=OBJECT_PROMPT_ABO):
    img_dir = f"{processed_dir}/images"
    mask_dir = f"{processed_dir}/masks"
    os.makedirs(mask_dir, exist_ok=True)
    names = sorted(os.listdir(img_dir))
    no_box = []
    for n in names:
        img = Image.open(f"{img_dir}/{n}").convert("RGB")
        W, H = img.size
        b, s = gdino_boxes(img, prompt, 0.3)
        if len(b) == 0:
            b, s = gdino_boxes(img, "furniture. object.", 0.22)
        keep = [bb for bb in b if 0.02 < ((bb[2]-bb[0])*(bb[3]-bb[1]))/(W*H) < 0.99]
        if not keep:
            no_box.append(n)
            mask = np.ones((H, W), bool)
        else:
            mask = sam_union(img, keep)
        Image.fromarray((mask*255).astype(np.uint8)).save(f"{mask_dir}/{os.path.splitext(n)[0]}.png")
    print(processed_dir, "-> масок:", len(os.listdir(mask_dir)), "| без бокса:", len(no_box), no_box[:5])

for name in ["raw_dense", "raw_N30", "raw_N20"]:
    pd = f"{D_ABO}/{name}_processed"
    if os.path.exists(f"{pd}/transforms.json"):
        make_masks(pd)

In [ ]:
def train_splatfacto(processed_dir, out_name, iters):
    cmd = (
        f"yes | TORCHDYNAMO_DISABLE=1 /content/nsenv/bin/ns-train splatfacto "
        f"--data {processed_dir} --max-num-iterations {iters} --steps-per-save 1000 "
        f"--vis tensorboard --viewer.quit-on-train-completion True "
        f"--output-dir {D_ABO}/outputs_{out_name} --pipeline.model.cull-alpha-thresh 0.1 "
        f"nerfstudio-data --masks-path masks"
    )
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(out_name, "-> returncode", r.returncode)
    print(r.stdout[-1500:])
    if r.returncode:
        print(r.stderr[-1500:])

train_splatfacto(f"{D_ABO}/raw_dense_processed", "dense", iters=7000)
train_splatfacto(f"{D_ABO}/raw_N30_processed", "n30", iters=3000)
train_splatfacto(f"{D_ABO}/raw_N20_processed", "n20", iters=3000)

In [ ]:
import glob

def export_and_mesh(out_name):
    cfgs = glob.glob(f"{D_ABO}/outputs_{out_name}/*/splatfacto/*/config.yml")
    if not cfgs:
        print(out_name, "-> config.yml не найден, тренировка не удалась")
        return
    cfg = sorted(cfgs)[-1]
    export_dir = f"{D_ABO}/exports_{out_name}"
    subprocess.run(f"/content/nsenv/bin/ns-export gaussian-splat --load-config {cfg} --output-dir {export_dir}", shell=True)

    script = f'''
import numpy as np
import open3d as o3d
import trimesh

pcd = o3d.io.read_point_cloud("{export_dir}/splat.ply")
pcd, _ = pcd.remove_statistical_outlier(nb_neighbors=20, std_ratio=2.0)
nn = o3d.geometry.KDTreeFlann(pcd)
step = max(1, len(pcd.points) // 2000)
d = []
for i in range(0, len(pcd.points), step):
    _, idx, dist2 = nn.search_knn_vector_3d(pcd.points[i], 2)
    d.append(np.sqrt(dist2[1]))
eps = 5 * float(np.median(d))
labels = np.array(pcd.cluster_dbscan(eps=eps, min_points=10))
if labels.max() >= 0:
    biggest = np.bincount(labels[labels >= 0]).argmax()
    pcd = pcd.select_by_index(np.where(labels == biggest)[0])
pcd.estimate_normals(o3d.geometry.KDTreeSearchParamHybrid(radius=0.1, max_nn=30))
pcd.orient_normals_consistent_tangent_plane(30)
mesh, dens = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(pcd, depth=9)
dens = np.asarray(dens)
mesh.remove_vertices_by_mask(dens <= np.quantile(dens, 0.1))
mesh.remove_degenerate_triangles(); mesh.remove_duplicated_vertices()
mesh.remove_duplicated_triangles(); mesh.remove_non_manifold_edges()
mesh.compute_triangle_normals()
out = "{export_dir}/chair_object.stl"
o3d.io.write_triangle_mesh(out, mesh)
m = trimesh.load(out)
m.fill_holes(); m.fix_normals(); m.process(validate=True)
m.export(out)
print("STL:", out, "| Ecken:", len(m.vertices), "| Dreiecke:", len(m.faces))
print("watertight:", m.is_watertight)
'''
    open(f"/content/mesh_abo_{out_name}.py", "w").write(script)
    r = subprocess.run(["/content/nsenv/bin/python", f"/content/mesh_abo_{out_name}.py"], capture_output=True, text=True)
    print(r.stdout)
    if r.returncode:
        print(r.stderr[-2000:])

for name in ["dense", "n30", "n20"]:
    export_and_mesh(name)

### 9b. Chamfer-дистанция против настоящей CAD-модели

Сравнение каждой реконструкции (dense/n30/n20) со скачанной ABO CAD-моделью
(`B07F2X8K62.glb`) — настоящий внешний ground truth вместо self-consistency proxy из
PROJECT_STATUS.md, раздел 5. Метод тот же, что и там (нормализация масштаба, перебор
поворотов 0-345° с шагом 15°, ICP-доводка).

In [ ]:
r = subprocess.run(["/content/nsenv/bin/pip", "install", "-q", "pygltflib"], capture_output=True, text=True)
print(r.returncode, r.stderr[-500:] if r.returncode else "pygltflib ok")

In [ ]:
script = '''
import numpy as np
import open3d as o3d
import trimesh

def load_pts(path, n=30000):
    m = trimesh.load(path, force="mesh")
    pts, _ = trimesh.sample.sample_surface(m, n)
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(pts)
    return pcd

def normalize(pcd):
    pts = np.asarray(pcd.points)
    c = pts.mean(0)
    pts = pts - c
    scale = np.linalg.norm(pts, axis=1).max()
    pts = pts / scale
    out = o3d.geometry.PointCloud()
    out.points = o3d.utility.Vector3dVector(pts)
    return out

def chamfer_vs_ref(ref, cand_path, label):
    cand = normalize(load_pts(cand_path))
    best = None
    for deg in range(0, 360, 15):
        R = o3d.geometry.get_rotation_matrix_from_axis_angle([0, 0, np.deg2rad(deg)])
        c2 = o3d.geometry.PointCloud(cand)
        c2.rotate(R, center=(0, 0, 0))
        reg = o3d.pipelines.registration.registration_icp(
            c2, ref, 0.05, np.eye(4),
            o3d.pipelines.registration.TransformationEstimationPointToPoint())
        if best is None or reg.inlier_rmse < best[0]:
            best = (reg.inlier_rmse, deg, reg.transformation, R)
    rmse, deg, T, R0 = best
    aligned = o3d.geometry.PointCloud(cand)
    aligned.rotate(R0, center=(0, 0, 0))
    aligned.transform(T)
    reg2 = o3d.pipelines.registration.registration_icp(
        aligned, ref, 0.03, np.eye(4),
        o3d.pipelines.registration.TransformationEstimationPointToPoint())
    aligned.transform(reg2.transformation)
    d1 = np.asarray(aligned.compute_point_cloud_distance(ref))
    d2 = np.asarray(ref.compute_point_cloud_distance(aligned))
    chamfer = d1.mean() + d2.mean()
    print(label, ": best angle", deg, "deg, ICP inlier RMSE %.4f" % rmse, ", Chamfer %.4f" % chamfer)
    return chamfer

ref_cad = normalize(load_pts("/content/abo_chair/cad/B07F2X8K62.glb"))
for name in ["dense", "n30", "n20"]:
    path = "/content/abo_chair/exports_" + name + "/chair_object.stl"
    import os as _os
    if _os.path.exists(path):
        chamfer_vs_ref(ref_cad, path, name + " vs. ABO-CAD")
    else:
        print(name, "-> STL нет, пропуск")
'''
open('/content/chamfer_vs_cad.py', 'w').write(script)
r = subprocess.run(['/content/nsenv/bin/python', '/content/chamfer_vs_cad.py'], capture_output=True, text=True)
print(r.stdout)
if r.returncode:
    print('STDERR:', r.stderr[-3000:])

In [ ]:
import shutil
os.makedirs('/content/drive/MyDrive/nerf_cache/abo_chair', exist_ok=True)
for name in ["dense", "n30", "n20"]:
    src_stl = f"{D_ABO}/exports_{name}/chair_object.stl"
    if os.path.exists(src_stl):
        shutil.copy(src_stl, f"/content/drive/MyDrive/nerf_cache/abo_chair/chair_object_{name}.stl")
print("сохранено в Drive:", os.listdir('/content/drive/MyDrive/nerf_cache/abo_chair'))

## 10. Повтор: n=20 sparse-view тест на truck (COLMAP нестабилен, пробуем ещё раз)

PROJECT_STATUS.md, раздел 8, пункт 3: предыдущая попытка n=20 зарегистрировала только 4/20
кадров (слишком мало для реконструкции) — COLMAP на n=20 недетерминирован, набор фото тот же
(`random.seed(0)` в ячейке 20/21), но само сопоставление признаков/RANSAC каждый раз идёт
по-разному. Ниже — до 5 автоматических попыток подряд, берём лучшую, и на ней считаем ту же
Chamfer-дистанцию, что и для n=30 (раздел 5).

Предполагается, что `/content/sparse_test/raw` уже содержит 20 фото (ячейка 20/21) и что
модели/функции Teil 4 (`gdino_boxes`, `sam_union`, `OBJECT_PROMPT`) уже выполнены.

In [ ]:
import json as _json, subprocess, os, shutil

RAW20 = '/content/sparse_test/raw'
PROCESSED20 = '/content/sparse_test/processed'
BEST20 = '/content/sparse_test/processed_best'

def try_colmap_once():
    subprocess.run(f'rm -rf {PROCESSED20}', shell=True)
    cmd = (
        f"QT_QPA_PLATFORM=offscreen /content/nsenv/bin/ns-process-data images "
        f"--data {RAW20} --output-dir {PROCESSED20} --matching-method exhaustive --no-gpu"
    )
    subprocess.run(cmd, shell=True, capture_output=True, text=True)
    tf = f'{PROCESSED20}/transforms.json'
    if os.path.exists(tf):
        return len(_json.load(open(tf))['frames'])
    return 0

best_n = -1
MAX_TRIES = 5
for attempt in range(1, MAX_TRIES + 1):
    n_reg = try_colmap_once()
    print(f"попытка {attempt}: зарегистрировано {n_reg}/20")
    if n_reg > best_n:
        best_n = n_reg
        if os.path.isdir(BEST20):
            shutil.rmtree(BEST20)
        shutil.copytree(PROCESSED20, BEST20)
    if n_reg >= 15:  # сопоставимо с n=30 (22/30 ~ 73%), дальше можно не пытаться
        print("достаточно кадров, останавливаемся")
        break

print(f"итог: лучшая попытка {best_n}/20 -> {BEST20}")
if best_n < 8:
    print("ВНИМАНИЕ: даже лучшая попытка слишком слабая для осмысленной реконструкции, "
          "смотри секцию 4b PROJECT_STATUS.md — увеличить MAX_TRIES или взять другой seed.")

In [ ]:
# маски: переиспользуем gdino_boxes/sam_union/OBJECT_PROMPT из Teil 4 ("a truck.")
img_dir = f'{BEST20}/images'
mask_dir = f'{BEST20}/masks'
os.makedirs(mask_dir, exist_ok=True)
names = sorted(os.listdir(img_dir))
no_box = []
for n in names:
    img = Image.open(f"{img_dir}/{n}").convert("RGB")
    W, H = img.size
    b, s = gdino_boxes(img, OBJECT_PROMPT, 0.3)
    if len(b) == 0:
        b, s = gdino_boxes(img, "object. vehicle.", 0.22)
    keep = [bb for bb in b if 0.02 < ((bb[2]-bb[0])*(bb[3]-bb[1]))/(W*H) < 0.99]
    if not keep:
        no_box.append(n)
        mask = np.ones((H, W), bool)
    else:
        mask = sam_union(img, keep)
    Image.fromarray((mask*255).astype(np.uint8)).save(f"{mask_dir}/{os.path.splitext(n)[0]}.png")
print("масок:", len(os.listdir(mask_dir)), "| без бокса:", len(no_box), no_box[:5])

In [ ]:
cmd = (
    f"yes | TORCHDYNAMO_DISABLE=1 /content/nsenv/bin/ns-train splatfacto "
    f"--data {BEST20} --max-num-iterations 3000 --steps-per-save 1000 "
    f"--vis tensorboard --viewer.quit-on-train-completion True "
    f"--output-dir outputs_sparse20 --pipeline.model.cull-alpha-thresh 0.1 "
    f"nerfstudio-data --masks-path masks"
)
r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
print("returncode:", r.returncode)
print(r.stdout[-1500:])
if r.returncode:
    print(r.stderr[-1500:])

In [ ]:
import glob

cfgs = glob.glob("outputs_sparse20/*/splatfacto/*/config.yml")
if not cfgs:
    print("config.yml не найден, тренировка не удалась")
else:
    cfg = sorted(cfgs)[-1]
    subprocess.run(f"/content/nsenv/bin/ns-export gaussian-splat --load-config {cfg} --output-dir exports/sparse20", shell=True)

    script = '''
import numpy as np
import open3d as o3d
import trimesh

pcd = o3d.io.read_point_cloud("exports/sparse20/splat.ply")
pcd, _ = pcd.remove_statistical_outlier(nb_neighbors=20, std_ratio=2.0)
nn = o3d.geometry.KDTreeFlann(pcd)
step = max(1, len(pcd.points) // 2000)
d = []
for i in range(0, len(pcd.points), step):
    _, idx, dist2 = nn.search_knn_vector_3d(pcd.points[i], 2)
    d.append(np.sqrt(dist2[1]))
eps = 5 * float(np.median(d))
labels = np.array(pcd.cluster_dbscan(eps=eps, min_points=10))
if labels.max() >= 0:
    biggest = np.bincount(labels[labels >= 0]).argmax()
    pcd = pcd.select_by_index(np.where(labels == biggest)[0])
pcd.estimate_normals(o3d.geometry.KDTreeSearchParamHybrid(radius=0.1, max_nn=30))
pcd.orient_normals_consistent_tangent_plane(30)
mesh, dens = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(pcd, depth=9)
dens = np.asarray(dens)
mesh.remove_vertices_by_mask(dens <= np.quantile(dens, 0.1))
mesh.remove_degenerate_triangles(); mesh.remove_duplicated_vertices()
mesh.remove_duplicated_triangles(); mesh.remove_non_manifold_edges()
mesh.compute_triangle_normals()
out = "exports/sparse20/sparse20_object.stl"
o3d.io.write_triangle_mesh(out, mesh)
m = trimesh.load(out)
m.fill_holes(); m.fix_normals(); m.process(validate=True)
m.export(out)
print("STL:", out, "| Ecken:", len(m.vertices), "| Dreiecke:", len(m.faces))
print("watertight:", m.is_watertight)
'''
    open('/content/mesh_sparse20.py', 'w').write(script)
    r = subprocess.run(['/content/nsenv/bin/python', '/content/mesh_sparse20.py'], capture_output=True, text=True)
    print(r.stdout)
    if r.returncode:
        print(r.stderr[-2000:])

In [ ]:
script = '''
import numpy as np
import open3d as o3d
import trimesh

def load_pts(path, n=30000):
    m = trimesh.load(path)
    pts, _ = trimesh.sample.sample_surface(m, n)
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(pts)
    return pcd

def normalize(pcd):
    pts = np.asarray(pcd.points)
    c = pts.mean(0)
    pts = pts - c
    scale = np.linalg.norm(pts, axis=1).max()
    pts = pts / scale
    out = o3d.geometry.PointCloud()
    out.points = o3d.utility.Vector3dVector(pts)
    return out

ref = normalize(load_pts("exports/truck_masked/truck_object_ref.stl"))
sparse = normalize(load_pts("exports/sparse20/sparse20_object.stl"))

best = None
for deg in range(0, 360, 15):
    R = o3d.geometry.get_rotation_matrix_from_axis_angle([0, 0, np.deg2rad(deg)])
    cand = o3d.geometry.PointCloud(sparse)
    cand.rotate(R, center=(0, 0, 0))
    reg = o3d.pipelines.registration.registration_icp(
        cand, ref, 0.05, np.eye(4),
        o3d.pipelines.registration.TransformationEstimationPointToPoint())
    if best is None or reg.inlier_rmse < best[0]:
        best = (reg.inlier_rmse, deg, reg.transformation, R)

rmse, deg, T, R0 = best
print("лучший начальный угол:", deg, "| rmse после ICP:", rmse)

sparse_aligned = o3d.geometry.PointCloud(sparse)
sparse_aligned.rotate(R0, center=(0, 0, 0))
sparse_aligned.transform(T)

reg2 = o3d.pipelines.registration.registration_icp(
    sparse_aligned, ref, 0.03, np.eye(4),
    o3d.pipelines.registration.TransformationEstimationPointToPoint())
sparse_aligned.transform(reg2.transformation)

d1 = np.asarray(sparse_aligned.compute_point_cloud_distance(ref))
d2 = np.asarray(ref.compute_point_cloud_distance(sparse_aligned))
chamfer = d1.mean() + d2.mean()
print(f"Chamfer distance n=20 (нормализовано на размер объекта): {chamfer:.4f}")
print(f"  -> sparse20 к ref: {d1.mean():.4f} (медиана {np.median(d1):.4f})")
print(f"  -> ref к sparse20: {d2.mean():.4f} (медиана {np.median(d2):.4f})")
'''
open('/content/chamfer20.py', 'w').write(script)
r = subprocess.run(['/content/nsenv/bin/python', '/content/chamfer20.py'], capture_output=True, text=True)
print(r.stdout)
if r.returncode:
    print('STDERR:', r.stderr[-3000:])

In [ ]:
import os, subprocess, sys

ABO_ITEM = "B07F2X8K62"
ABO_SPIN_ID = "f8022078"
ABO_SPIN_PREFIX = ABO_SPIN_ID[:2]
ABO_N_FRAMES = 72
D_ABO = "/content/abo_chair"
os.makedirs(f"{D_ABO}/raw", exist_ok=True)

base = "https://amazon-berkeley-objects.s3.amazonaws.com"
for az in range(ABO_N_FRAMES):
    fn = f"{ABO_SPIN_ID}_{az:02d}.jpg"
    url = f"{base}/spins/original/{ABO_SPIN_PREFIX}/{ABO_SPIN_ID}/{fn}"
    out = f"{D_ABO}/raw/{fn}"
    if not os.path.exists(out):
        r = subprocess.run(["curl", "-s", "-L", "-w", "%{http_code}", url, "-o", out], capture_output=True, text=True)
        print(az, fn, "http", r.stdout.strip())
        sys.stdout.flush()
print("фото скачано:", len(os.listdir(f"{D_ABO}/raw")))

cad_dir = f"{D_ABO}/cad"
os.makedirs(cad_dir, exist_ok=True)
cad_url = f"{base}/3dmodels/original/{ABO_ITEM[-1]}/{ABO_ITEM}.glb"
r = subprocess.run(["curl", "-s", "-L", "-w", "%{http_code}", cad_url, "-o", f"{cad_dir}/{ABO_ITEM}.glb"], capture_output=True, text=True)
print("CAD http", r.stdout.strip(), "->", os.listdir(cad_dir))


0 f8022078_00.jpg http 200
1 f8022078_01.jpg http 200
2 f8022078_02.jpg http 200
3 f8022078_03.jpg http 200
4 f8022078_04.jpg http 200
5 f8022078_05.jpg http 200
6 f8022078_06.jpg http 200
7 f8022078_07.jpg http 200
8 f8022078_08.jpg http 200
9 f8022078_09.jpg http 200
10 f8022078_10.jpg http 200
11 f8022078_11.jpg http 200
12 f8022078_12.jpg http 200
13 f8022078_13.jpg http 200
14 f8022078_14.jpg http 200
15 f8022078_15.jpg http 200
16 f8022078_16.jpg http 200
17 f8022078_17.jpg http 200
18 f8022078_18.jpg http 200
19 f8022078_19.jpg http 200
20 f8022078_20.jpg http 200
21 f8022078_21.jpg http 200
22 f8022078_22.jpg http 200
23 f8022078_23.jpg http 200
24 f8022078_24.jpg http 200
25 f8022078_25.jpg http 200
26 f8022078_26.jpg http 200
27 f8022078_27.jpg http 200
28 f8022078_28.jpg http 200
29 f8022078_29.jpg http 200
30 f8022078_30.jpg http 200
31 f8022078_31.jpg http 200
32 f8022078_32.jpg http 200
33 f8022078_33.jpg http 200
34 f8022078_34.jpg http 200
35 f8022078_35.jpg http 200
36

In [ ]:
import shutil
import numpy as np

def make_subset(n, raw_dir=f"{D_ABO}/raw", out_name=None):
    out_name = out_name or f"raw_N{n}"
    out_dir = f"{D_ABO}/{out_name}"
    if os.path.isdir(out_dir):
        shutil.rmtree(out_dir)
    os.makedirs(out_dir)
    names = sorted(os.listdir(raw_dir))
    idx = sorted(set(np.linspace(0, len(names) - 1, n).round().astype(int).tolist()))
    for i in idx:
        shutil.copy(f"{raw_dir}/{names[i]}", out_dir)
    print(out_name, "->", len(os.listdir(out_dir)), "фото")

make_subset(ABO_N_FRAMES, out_name="raw_dense")  # полная turntable-серия как референс
make_subset(30)
make_subset(20)


raw_dense -> 72 фото
raw_N30 -> 30 фото
raw_N20 -> 20 фото


In [ ]:
!apt-get -qq update
!apt-get -qq install -y colmap
!colmap -h | head -3


W: https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt/trusted.gpg), see the DEPRECATION section in apt-key(8) for details.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Extracting templates from packages: 100%
Selecting previously unselected package libevdev2:amd64.
(Reading database ... 126952 files and directories currently installed.)
Preparing to unpack .../00-libevdev2_1.13.1+dfsg-1build1_amd64.deb ...
Unpacking libevdev2:amd64 (1.13.1+dfsg-1build1) ...
Selecting previously unselected package at-spi2-common.
Preparing to unpack .../01-at-spi2-common_2.52.0-1build1_all.deb ...
Unpacking at-spi2-common (2.52.0-1build1) ...
Selecting previously unselected package libatspi2.0-0t64:amd64.
Preparing to unpack .../02-libatspi2.0-0t64_2.52.0-1build1_amd64.deb .

In [ ]:
def run_colmap(subset_dir_name):
    raw = f"{D_ABO}/{subset_dir_name}"
    processed = f"{D_ABO}/{subset_dir_name}_processed"
    subprocess.run(f"rm -rf {processed}", shell=True)
    cmd = (
        f"QT_QPA_PLATFORM=offscreen /content/nsenv/bin/ns-process-data images "
        f"--data {raw} --output-dir {processed} --matching-method exhaustive --no-gpu"
    )
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(subset_dir_name, "-> returncode", r.returncode)
    print(r.stdout[-1500:])
    if r.returncode:
        print(r.stderr[-1500:])

for name in ["raw_dense", "raw_N30", "raw_N20"]:
    run_colmap(name)


raw_dense -> returncode 127

/bin/sh: 1: /content/nsenv/bin/ns-process-data: not found

raw_N30 -> returncode 127

/bin/sh: 1: /content/nsenv/bin/ns-process-data: not found

raw_N20 -> returncode 127

/bin/sh: 1: /content/nsenv/bin/ns-process-data: not found



In [ ]:
!ls -la /content/nsenv/bin/ 2>&1 | head -20
!ls -la /content/ 2>&1


ls: cannot access '/content/nsenv/bin/': No such file or directory
total 20
drwxr-xr-x 1 root root 4096 Sep 30 16:51 .
drwxr-xr-x 1 root root 4096 Sep 30 16:39 ..
drwxr-xr-x 7 root root 4096 Sep 30 16:55 abo_chair
drwxr-xr-x 4 root root 4096 Sep  4 13:32 .config
drwxr-xr-x 1 root root 4096 Sep  4 13:32 sample_data


In [ ]:
%%bash
set -e
NSPY=/content/nsenv/bin/python

if [ -x /content/nsenv/bin/ns-train ]; then
  echo '=== venv уже собран, проверка ==='
  $NSPY -c "import numpy,torch,gsplat; from nerfstudio.models.splatfacto import SplatfactoModel; print('numpy',numpy.__version__,'| torch',torch.__version__,'| gsplat',gsplat.__version__,'| cuda',torch.cuda.is_available(),'| splatfacto OK')"
  exit 0
fi

pip install -q uv
uv python install 3.10
uv venv --python 3.10 /content/nsenv

uv pip install --python $NSPY "torch==2.1.2" "torchvision==0.16.2" --index-url https://download.pytorch.org/whl/cu118
uv pip install --python $NSPY "setuptools<70" wheel
uv pip install --python $NSPY "gsplat==1.4.0+pt21cu118" --index-url https://docs.gsplat.studio/whl/pt21cu118 --extra-index-url https://pypi.org/simple --index-strategy unsafe-best-match
uv pip install --python $NSPY "nerfstudio==1.1.5" open3d trimesh
uv pip install --python $NSPY "numpy==1.26.4"

echo '=== проверка ==='
$NSPY -c "import numpy,torch,gsplat; from gsplat.cuda._backend import _C; from nerfstudio.models.splatfacto import SplatfactoModel; print('numpy',numpy.__version__,'| torch',torch.__version__,'| gsplat',gsplat.__version__,'| cuda',torch.cuda.is_available(),'| splatfacto OK')"
/content/nsenv/bin/ns-train --help >/dev/null 2>&1 && echo 'ns-train: OK' || echo 'ns-train: FAIL'


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.4/20.4 MB 86.1 MB/s eta 0:00:00
=== проверка ===
numpy 1.26.4 | torch 2.1.2+cu118 | gsplat 1.4.0+pt21cu118 | cuda True | splatfacto OK
ns-train: OK


Installed Python 3.10.21 in 1.97s
 + cpython-3.10.21-linux-x86_64-gnu (python3.10)
Using CPython 3.10.21
Creating virtual environment at: nsenv
Activate with: source nsenv/bin/activate
Using Python 3.10.21 environment at: nsenv
Resolved 18 packages in 2.84s
Prepared 18 packages in 48.99s
Installed 18 packages in 260ms
 + certifi==2022.12.7
 + charset-normalizer==2.1.1
 + filelock==3.32.3
 + fsspec==2026.7.0
 + idna==3.4
 + jinja2==3.1.6
 + markupsafe==3.0.3
 + mpmath==1.3.0
 + networkx==3.4.2
 + numpy==2.2.6
 + pillow==12.3.0
 + requests==2.28.1
 + sympy==1.14.0
 + torch==2.1.2+cu118
 + torchvision==0.16.2+cu118
 + triton==2.1.0
 + typing-extensions==4.16.0
 + urllib3==1.26.13
Using Python 3.10.21 environment at: nsenv
Resolved 3 packages in 103ms
Prepared 3 packages in 29ms
Installed 3 packages in 6ms
 + packaging==26.3
 + setuptools==69.5.1
 + wheel==0.48.0
Using Python 3.10.21 environment at: nsenv
Resolved 19 packages in 2.57s
Prepared 8 packages in 516ms
Installed 8 packages in 57

In [ ]:
print(gd_model.device)


NameError: name 'gd_model' is not defined

In [ ]:
!pip install -q -U transformers
import torch, numpy as np
from PIL import Image
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection, Sam2Model, Sam2Processor

dev = "cuda"
OBJECT_PROMPT = "a truck."  # текстовое описание объекта пользователя — заменить под реальный сценарий
GD, SAM = "IDEA-Research/grounding-dino-base", "facebook/sam2-hiera-large"
gd_proc = AutoProcessor.from_pretrained(GD)
gd_model = AutoModelForZeroShotObjectDetection.from_pretrained(GD).to(dev).eval()
sam_proc = Sam2Processor.from_pretrained(SAM)
sam_model = Sam2Model.from_pretrained(SAM).to(dev).eval()
print("модели ок (SAM2)")

def gdino_boxes(img, text=OBJECT_PROMPT, thr=0.3):
    inp = gd_proc(images=img, text=text, return_tensors="pt").to(dev)
    with torch.no_grad():
        out = gd_model(**inp)
    tgt = [img.size[::-1]]
    try:
        r = gd_proc.post_process_grounded_object_detection(out, inp.input_ids, threshold=thr, text_threshold=0.25, target_sizes=tgt)[0]
    except TypeError:
        r = gd_proc.post_process_grounded_object_detection(out, inp.input_ids, box_threshold=thr, text_threshold=0.25, target_sizes=tgt)[0]
    return r["boxes"].cpu().numpy(), r["scores"].cpu().numpy()


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 113.1 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 58.1 MB/s eta 0:00:00


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


preprocessor_config.json:   0%|          | 0.00/457 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.74k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.24k [00:00<?, ?B/s]

...

[transformers] You are using a model of type `sam2_video` to instantiate a model of type `sam2`. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.


model.safetensors: reconstructing file:   0%|          |  0.00B /  898MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/741 [00:00<?, ?it/s]

модели ок (SAM2)


In [ ]:
!colmap -h | head -3


COLMAP 3.9.1 -- Structure-from-Motion and Multi-View Stereo
(Commit Unknown on Unknown without CUDA)



In [ ]:
!pip install -q -U transformers
import torch, numpy as np
from PIL import Image
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection, Sam2Model, Sam2Processor

dev = "cuda"
OBJECT_PROMPT = "a truck."
GD, SAM = "IDEA-Research/grounding-dino-base", "facebook/sam2-hiera-large"
gd_proc = AutoProcessor.from_pretrained(GD)
gd_model = AutoModelForZeroShotObjectDetection.from_pretrained(GD).to(dev).eval()
sam_proc = Sam2Processor.from_pretrained(SAM)
sam_model = Sam2Model.from_pretrained(SAM).to(dev).eval()
print("модели ок (SAM2)")

def gdino_boxes(img, text=OBJECT_PROMPT, thr=0.3):
    inp = gd_proc(images=img, text=text, return_tensors="pt").to(dev)
    with torch.no_grad():
        out = gd_model(**inp)
    tgt = [img.size[::-1]]
    try:
        r = gd_proc.post_process_grounded_object_detection(out, inp.input_ids, threshold=thr, text_threshold=0.25, target_sizes=tgt)[0]
    except TypeError:
        r = gd_proc.post_process_grounded_object_detection(out, inp.input_ids, box_threshold=thr, text_threshold=0.25, target_sizes=tgt)[0]
    return r["boxes"].cpu().numpy(), r["scores"].cpu().numpy()


Loading weights:   0%|          | 0/1182 [00:00<?, ?it/s]

[transformers] You are using a model of type `sam2_video` to instantiate a model of type `sam2`. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.


Loading weights:   0%|          | 0/741 [00:00<?, ?it/s]

модели ок (SAM2)


In [ ]:
!colmap -h | head -3


COLMAP 3.9.1 -- Structure-from-Motion and Multi-View Stereo
(Commit Unknown on Unknown without CUDA)



In [ ]:
import os, subprocess

D_ABO = "/content/abo_chair"  # на случай если переменная тоже потерялась

def run_colmap(subset_dir_name):
    raw = f"{D_ABO}/{subset_dir_name}"
    processed = f"{D_ABO}/{subset_dir_name}_processed"
    subprocess.run(f"rm -rf {processed}", shell=True)
    cmd = (
        f"QT_QPA_PLATFORM=offscreen /content/nsenv/bin/ns-process-data images "
        f"--data {raw} --output-dir {processed} --matching-method exhaustive --no-gpu"
    )
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(subset_dir_name, "-> returncode", r.returncode)
    print(r.stdout[-1500:])
    if r.returncode:
        print(r.stderr[-1500:])

for name in ["raw_dense", "raw_N30", "raw_N20"]:
    run_colmap(name)


raw_dense -> returncode 0
[17:10:49] 🎉 Done copying images with prefix 'frame_'.                                        process_data_utils.py:348
[17:12:19] 🎉 Done extracting COLMAP features.                                                       colmap_utils.py:137
[17:13:26] 🎉 Done matching COLMAP features.                                                         colmap_utils.py:151
[17:13:27] 🎉 Done COLMAP bundle adjustment.                                                         colmap_utils.py:173
           🎉 Done refining intrinsics.                                                              colmap_utils.py:184
           🎉 🎉 🎉 All DONE 🎉 🎉 🎉                                                images_to_nerfstudio_dataset.py:135
           Starting with 72 images                                                   images_to_nerfstudio_dataset.py:138
           Colmap matched 2 images                                                   images_to_nerfstudio_dataset.py:138
           COLMAP

In [ ]:
def run_colmap_seq(subset_dir_name, matching="sequential"):
    raw = f"{D_ABO}/{subset_dir_name}"
    processed = f"{D_ABO}/{subset_dir_name}_processed"
    subprocess.run(f"rm -rf {processed}", shell=True)
    cmd = (
        f"QT_QPA_PLATFORM=offscreen /content/nsenv/bin/ns-process-data images "
        f"--data {raw} --output-dir {processed} --matching-method {matching} --no-gpu"
    )
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(subset_dir_name, matching, "-> returncode", r.returncode)
    print(r.stdout[-2000:])
    if r.returncode:
        print(r.stderr[-1500:])

run_colmap_seq("raw_dense", "sequential")


raw_dense sequential -> returncode 0
[17:17:22] 🎉 Done copying images with prefix 'frame_'.                                        process_data_utils.py:348
[17:18:53] 🎉 Done extracting COLMAP features.                                                       colmap_utils.py:137
[17:19:11] 🎉 Done matching COLMAP features.                                                         colmap_utils.py:151
[17:19:12] 🎉 Done COLMAP bundle adjustment.                                                         colmap_utils.py:173
           🎉 Done refining intrinsics.                                                              colmap_utils.py:184
           🎉 🎉 🎉 All DONE 🎉 🎉 🎉                                                images_to_nerfstudio_dataset.py:135
           Starting with 72 images                                                   images_to_nerfstudio_dataset.py:138
           Colmap matched 2 images                                                   images_to_nerfstudio_dataset.py:138
      

In [ ]:
import numpy as np, json, os, shutil
from PIL import Image

def pose_spherical(theta_deg, phi_deg, radius):
    def trans_t(t):
        return np.array([[1,0,0,0],[0,1,0,0],[0,0,1,t],[0,0,0,1]], dtype=float)
    def rot_phi(p):
        p = np.deg2rad(p)
        return np.array([[1,0,0,0],[0,np.cos(p),-np.sin(p),0],[0,np.sin(p),np.cos(p),0],[0,0,0,1]], dtype=float)
    def rot_theta(t):
        t = np.deg2rad(t)
        return np.array([[np.cos(t),0,-np.sin(t),0],[0,1,0,0],[np.sin(t),0,np.cos(t),0],[0,0,0,1]], dtype=float)
    c2w = trans_t(radius)
    c2w = rot_phi(phi_deg) @ c2w
    c2w = rot_theta(theta_deg) @ c2w
    c2w = np.array([[-1,0,0,0],[0,0,1,0],[0,1,0,0],[0,0,0,1]], dtype=float) @ c2w
    return c2w

def build_synthetic_dataset(subset_dir_name, radius=2.5, elevation_deg=-15.0, fov_deg=40.0):
    raw_dir = f"{D_ABO}/{subset_dir_name}"
    names = sorted(os.listdir(raw_dir))
    w, h = Image.open(f"{raw_dir}/{names[0]}").size

    out_dir = f"{D_ABO}/{subset_dir_name}_synth"
    img_out = f"{out_dir}/images"
    os.makedirs(img_out, exist_ok=True)

    fov_x = np.deg2rad(fov_deg)
    fl = 0.5 * w / np.tan(0.5 * fov_x)

    frames = []
    for n in names:
        az = int(n.split('_')[1].split('.')[0])
        theta = az * 5.0
        c2w = pose_spherical(theta, elevation_deg, radius)
        dst = f"{img_out}/{n}"
        if not os.path.exists(dst):
            shutil.copy(f"{raw_dir}/{n}", dst)
        frames.append({"file_path": f"images/{n}", "transform_matrix": c2w.tolist()})

    out = {
        "fl_x": fl, "fl_y": fl,
        "cx": w / 2, "cy": h / 2,
        "w": w, "h": h,
        "camera_model": "OPENCV",
        "k1": 0.0, "k2": 0.0, "p1": 0.0, "p2": 0.0,
        "frames": frames,
    }
    with open(f"{out_dir}/transforms.json", "w") as f:
        json.dump(out, f, indent=2)
    print(subset_dir_name, "-> синтетические позы:", len(frames), "->", out_dir)

for name in ["raw_dense", "raw_N30", "raw_N20"]:
    build_synthetic_dataset(name)


raw_dense -> синтетические позы: 72 -> /content/abo_chair/raw_dense_synth
raw_N30 -> синтетические позы: 30 -> /content/abo_chair/raw_N30_synth
raw_N20 -> синтетические позы: 20 -> /content/abo_chair/raw_N20_synth


In [ ]:
OBJECT_PROMPT_ABO = "a chair."

def make_masks(processed_dir, prompt=OBJECT_PROMPT_ABO):
    img_dir = f"{processed_dir}/images"
    mask_dir = f"{processed_dir}/masks"
    os.makedirs(mask_dir, exist_ok=True)
    names = sorted(os.listdir(img_dir))
    no_box = []
    for n in names:
        img = Image.open(f"{img_dir}/{n}").convert("RGB")
        W, H = img.size
        b, s = gdino_boxes(img, prompt, 0.3)
        if len(b) == 0:
            b, s = gdino_boxes(img, "furniture. object.", 0.22)
        keep = [bb for bb in b if 0.02 < ((bb[2]-bb[0])*(bb[3]-bb[1]))/(W*H) < 0.99]
        if not keep:
            no_box.append(n)
            mask = np.ones((H, W), bool)
        else:
            mask = sam_union(img, keep)
        Image.fromarray((mask*255).astype(np.uint8)).save(f"{mask_dir}/{os.path.splitext(n)[0]}.png")
    print(processed_dir, "-> масок:", len(os.listdir(mask_dir)), "| без бокса:", len(no_box), no_box[:5])

for name in ["raw_dense", "raw_N30", "raw_N20"]:
    make_masks(f"{D_ABO}/{name}_synth")


NameError: name 'sam_union' is not defined

In [ ]:
_pp = getattr(sam_proc, "post_process_masks", None) or sam_proc.image_processor.post_process_masks

def sam_union(img, boxes):
    inp = sam_proc(img, input_boxes=[[list(map(float, b)) for b in boxes]], return_tensors="pt").to(dev)
    with torch.no_grad():
        out = sam_model(**inp, multimask_output=True)
    masks = _pp(out.pred_masks.cpu(), inp["original_sizes"].cpu())[0]
    m = masks.numpy()
    iou = out.iou_scores.cpu().numpy()[0]
    sel = np.stack([m[i, iou[i].argmax()] for i in range(m.shape[0])])
    return sel.any(0)

print("sam_union определена")


sam_union определена


In [ ]:
OBJECT_PROMPT_ABO = "a chair."

def make_masks(processed_dir, prompt=OBJECT_PROMPT_ABO):
    img_dir = f"{processed_dir}/images"
    mask_dir = f"{processed_dir}/masks"
    os.makedirs(mask_dir, exist_ok=True)
    names = sorted(os.listdir(img_dir))
    no_box = []
    for n in names:
        img = Image.open(f"{img_dir}/{n}").convert("RGB")
        W, H = img.size
        b, s = gdino_boxes(img, prompt, 0.3)
        if len(b) == 0:
            b, s = gdino_boxes(img, "furniture. object.", 0.22)
        keep = [bb for bb in b if 0.02 < ((bb[2]-bb[0])*(bb[3]-bb[1]))/(W*H) < 0.99]
        if not keep:
            no_box.append(n)
            mask = np.ones((H, W), bool)
        else:
            mask = sam_union(img, keep)
        Image.fromarray((mask*255).astype(np.uint8)).save(f"{mask_dir}/{os.path.splitext(n)[0]}.png")
    print(processed_dir, "-> масок:", len(os.listdir(mask_dir)), "| без бокса:", len(no_box), no_box[:5])

for name in ["raw_dense", "raw_N30", "raw_N20"]:
    make_masks(f"{D_ABO}/{name}_synth")


/content/abo_chair/raw_dense_synth -> масок: 72 | без бокса: 0 []
/content/abo_chair/raw_N30_synth -> масок: 30 | без бокса: 0 []
/content/abo_chair/raw_N20_synth -> масок: 20 | без бокса: 0 []


In [ ]:
def train_splatfacto(processed_dir, out_name, iters):
    cmd = (
        f"yes | TORCHDYNAMO_DISABLE=1 /content/nsenv/bin/ns-train splatfacto "
        f"--data {processed_dir} --max-num-iterations {iters} --steps-per-save 1000 "
        f"--vis tensorboard --viewer.quit-on-train-completion True "
        f"--output-dir {D_ABO}/outputs_{out_name} --pipeline.model.cull-alpha-thresh 0.1 "
        f"nerfstudio-data --masks-path masks"
    )
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(out_name, "-> returncode", r.returncode)
    print(r.stdout[-2000:])
    if r.returncode:
        print(r.stderr[-2000:])

train_splatfacto(f"{D_ABO}/raw_dense_synth", "dense", iters=7000)
train_splatfacto(f"{D_ABO}/raw_N30_synth", "n30", iters=3000)
train_splatfacto(f"{D_ABO}/raw_N20_synth", "n20", iters=3000)


dense -> returncode 2

                                                      │
│   masks (applied to /content/nsenv/bin/ns-train splatfacto nerfstudio-data)  │
│                                                                              │
│ Arguments are applied to the directly preceding subcommand, so ordering can  │
│ matter.                                                                      │
│ ──────────────────────────────────────────────────────────────────────────── │
│ Perhaps you meant:                                                           │
│     --masks-path {None}|PATH                                                 │
│         Path to masks directory. If not set, masks are not loaded. (default: │
│         None)                                                                │
│         in /content/nsenv/bin/ns-train splatfacto colmap --help              │
│         in /content/nsenv/bin/ns-train splatfacto colmap --help              │
│         in /content/nsenv/bi

In [ ]:
def add_mask_paths(out_name):
    d = f"{D_ABO}/{out_name}_synth"
    tf = f"{d}/transforms.json"
    data = json.load(open(tf))
    for fr in data["frames"]:
        fname = os.path.basename(fr["file_path"])
        stem = os.path.splitext(fname)[0]
        mask_file = f"masks/{stem}.png"
        if os.path.exists(f"{d}/{mask_file}"):
            fr["mask_path"] = mask_file
    json.dump(data, open(tf, "w"), indent=2)
    n_with_mask = sum(1 for fr in data["frames"] if "mask_path" in fr)
    print(out_name, "-> масок привязано:", n_with_mask, "/", len(data["frames"]))

for name in ["raw_dense", "raw_N30", "raw_N20"]:
    add_mask_paths(name)


raw_dense -> масок привязано: 72 / 72
raw_N30 -> масок привязано: 30 / 30
raw_N20 -> масок привязано: 20 / 20


In [ ]:
def train_splatfacto(processed_dir, out_name, iters):
    cmd = (
        f"yes | TORCHDYNAMO_DISABLE=1 /content/nsenv/bin/ns-train splatfacto "
        f"--data {processed_dir} --max-num-iterations {iters} --steps-per-save 1000 "
        f"--vis tensorboard --viewer.quit-on-train-completion True "
        f"--output-dir {D_ABO}/outputs_{out_name} --pipeline.model.cull-alpha-thresh 0.1 "
        f"nerfstudio-data"
    )
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(out_name, "-> returncode", r.returncode)
    print(r.stdout[-2000:])
    if r.returncode:
        print(r.stderr[-2000:])

train_splatfacto(f"{D_ABO}/raw_dense_synth", "dense", iters=7000)
train_splatfacto(f"{D_ABO}/raw_N30_synth", "n30", iters=3000)
train_splatfacto(f"{D_ABO}/raw_N20_synth", "n20", iters=3000)


dense -> returncode 0
  5 s, 299.291 ms      17.95 M               
6920 (98.86%)       58.342 ms            4 s, 667.358 ms      18.08 M               
6930 (99.00%)       44.580 ms            3 s, 120.602 ms      18.85 M               
6940 (99.14%)       44.153 ms            2 s, 649.184 ms      19.03 M               
6950 (99.29%)       43.976 ms            2 s, 198.818 ms      19.10 M               
6960 (99.43%)       44.251 ms            1 s, 770.026 ms      18.99 M               
6970 (99.57%)       44.274 ms            1 s, 328.227 ms      18.98 M               
6980 (99.71%)       44.253 ms            885.068 ms           18.99 M               
6990 (99.86%)       44.466 ms            444.663 ms           18.90 M               
6999 (99.99%)                                                                       
╭────────────────────────────────────────────── 🎉 Training Finished 🎉 ───────────────────────────────────────────────╮
│                        ╷                      

In [1]:
import os
print("abo_chair:", os.path.exists('/content/abo_chair'))
print("venv:", os.path.exists('/content/nsenv/bin/ns-train'))
print("dense config:", os.path.exists('/content/abo_chair/outputs_dense'))
print("n30 config:", os.path.exists('/content/abo_chair/outputs_n30'))
print("n20 config:", os.path.exists('/content/abo_chair/outputs_n20'))


abo_chair: False
venv: False
dense config: False
n30 config: False
n20 config: False


In [2]:
%%bash
set -e
NSPY=/content/nsenv/bin/python

if [ -x /content/nsenv/bin/ns-train ]; then
  echo '=== venv уже собран, проверка ==='
  $NSPY -c "import numpy,torch,gsplat; from nerfstudio.models.splatfacto import SplatfactoModel; print('numpy',numpy.__version__,'| torch',torch.__version__,'| gsplat',gsplat.__version__,'| cuda',torch.cuda.is_available(),'| splatfacto OK')"
  exit 0
fi

pip install -q uv
uv python install 3.10
uv venv --python 3.10 /content/nsenv

uv pip install --python $NSPY "torch==2.1.2" "torchvision==0.16.2" --index-url https://download.pytorch.org/whl/cu118
uv pip install --python $NSPY "setuptools<70" wheel
uv pip install --python $NSPY "gsplat==1.4.0+pt21cu118" --index-url https://docs.gsplat.studio/whl/pt21cu118 --extra-index-url https://pypi.org/simple --index-strategy unsafe-best-match
uv pip install --python $NSPY "nerfstudio==1.1.5" open3d trimesh
uv pip install --python $NSPY "numpy==1.26.4"

echo '=== проверка ==='
$NSPY -c "import numpy,torch,gsplat; from gsplat.cuda._backend import _C; from nerfstudio.models.splatfacto import SplatfactoModel; print('numpy',numpy.__version__,'| torch',torch.__version__,'| gsplat',gsplat.__version__,'| cuda',torch.cuda.is_available(),'| splatfacto OK')"
/content/nsenv/bin/ns-train --help >/dev/null 2>&1 && echo 'ns-train: OK' || echo 'ns-train: FAIL'


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.4/20.4 MB 76.0 MB/s eta 0:00:00
=== проверка ===
numpy 1.26.4 | torch 2.1.2+cu118 | gsplat 1.4.0+pt21cu118 | cuda True | splatfacto OK
ns-train: OK


Installed Python 3.10.21 in 1.49s
 + cpython-3.10.21-linux-x86_64-gnu (python3.10)
Using CPython 3.10.21
Creating virtual environment at: nsenv
Activate with: source nsenv/bin/activate
Using Python 3.10.21 environment at: nsenv
Resolved 18 packages in 871ms
Prepared 18 packages in 46.90s
Installed 18 packages in 379ms
 + certifi==2022.12.7
 + charset-normalizer==2.1.1
 + filelock==3.32.3
 + fsspec==2026.7.0
 + idna==3.4
 + jinja2==3.1.6
 + markupsafe==3.0.3
 + mpmath==1.3.0
 + networkx==3.4.2
 + numpy==2.2.6
 + pillow==12.3.0
 + requests==2.28.1
 + sympy==1.14.0
 + torch==2.1.2+cu118
 + torchvision==0.16.2+cu118
 + triton==2.1.0
 + typing-extensions==4.16.0
 + urllib3==1.26.13
Using Python 3.10.21 environment at: nsenv
Resolved 3 packages in 192ms
Prepared 3 packages in 59ms
Installed 3 packages in 13ms
 + packaging==26.3
 + setuptools==69.5.1
 + wheel==0.48.0
Using Python 3.10.21 environment at: nsenv
Resolved 19 packages in 1.30s
Prepared 8 packages in 500ms
Installed 8 packages in 3

In [3]:
!pip install -q -U transformers
import torch, numpy as np
from PIL import Image
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection, Sam2Model, Sam2Processor

dev = "cuda"
OBJECT_PROMPT = "a truck."
GD, SAM = "IDEA-Research/grounding-dino-base", "facebook/sam2-hiera-large"
gd_proc = AutoProcessor.from_pretrained(GD)
gd_model = AutoModelForZeroShotObjectDetection.from_pretrained(GD).to(dev).eval()
sam_proc = Sam2Processor.from_pretrained(SAM)
sam_model = Sam2Model.from_pretrained(SAM).to(dev).eval()
print("модели ок (SAM2)")

def gdino_boxes(img, text=OBJECT_PROMPT, thr=0.3):
    inp = gd_proc(images=img, text=text, return_tensors="pt").to(dev)
    with torch.no_grad():
        out = gd_model(**inp)
    tgt = [img.size[::-1]]
    try:
        r = gd_proc.post_process_grounded_object_detection(out, inp.input_ids, threshold=thr, text_threshold=0.25, target_sizes=tgt)[0]
    except TypeError:
        r = gd_proc.post_process_grounded_object_detection(out, inp.input_ids, box_threshold=thr, text_threshold=0.25, target_sizes=tgt)[0]
    return r["boxes"].cpu().numpy(), r["scores"].cpu().numpy()

_pp = getattr(sam_proc, "post_process_masks", None) or sam_proc.image_processor.post_process_masks

def sam_union(img, boxes):
    inp = sam_proc(img, input_boxes=[[list(map(float, b)) for b in boxes]], return_tensors="pt").to(dev)
    with torch.no_grad():
        out = sam_model(**inp, multimask_output=True)
    masks = _pp(out.pred_masks.cpu(), inp["original_sizes"].cpu())[0]
    m = masks.numpy()
    iou = out.iou_scores.cpu().numpy()[0]
    sel = np.stack([m[i, iou[i].argmax()] for i in range(m.shape[0])])
    return sel.any(0)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 113.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 59.5 MB/s eta 0:00:00


preprocessor_config.json:   0%|          | 0.00/457 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.74k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.24k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

...

[transformers] You are using a model of type `sam2_video` to instantiate a model of type `sam2`. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.


model.safetensors: reconstructing file:   0%|          |  0.00B /  898MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/741 [00:00<?, ?it/s]

модели ок (SAM2)


In [4]:
import os, subprocess

ABO_ITEM = "B07F2X8K62"
ABO_SPIN_ID = "f8022078"
ABO_SPIN_PREFIX = ABO_SPIN_ID[:2]
ABO_N_FRAMES = 72
D_ABO = "/content/abo_chair"
os.makedirs(f"{D_ABO}/raw", exist_ok=True)

base = "https://amazon-berkeley-objects.s3.amazonaws.com"
for az in range(ABO_N_FRAMES):
    fn = f"{ABO_SPIN_ID}_{az:02d}.jpg"
    url = f"{base}/spins/original/{ABO_SPIN_PREFIX}/{ABO_SPIN_ID}/{fn}"
    out = f"{D_ABO}/raw/{fn}"
    if not os.path.exists(out):
        subprocess.run(["curl", "-s", "-L", url, "-o", out], check=True)
print("фото скачано:", len(os.listdir(f"{D_ABO}/raw")))

cad_dir = f"{D_ABO}/cad"
os.makedirs(cad_dir, exist_ok=True)
cad_url = f"{base}/3dmodels/original/{ABO_ITEM[-1]}/{ABO_ITEM}.glb"
subprocess.run(["curl", "-s", "-L", cad_url, "-o", f"{cad_dir}/{ABO_ITEM}.glb"], check=True)
print("CAD скачан:", os.listdir(cad_dir))


фото скачано: 72
CAD скачан: ['B07F2X8K62.glb']


In [5]:
import shutil
import numpy as np

def make_subset(n, raw_dir=f"{D_ABO}/raw", out_name=None):
    out_name = out_name or f"raw_N{n}"
    out_dir = f"{D_ABO}/{out_name}"
    if os.path.isdir(out_dir):
        shutil.rmtree(out_dir)
    os.makedirs(out_dir)
    names = sorted(os.listdir(raw_dir))
    idx = sorted(set(np.linspace(0, len(names) - 1, n).round().astype(int).tolist()))
    for i in idx:
        shutil.copy(f"{raw_dir}/{names[i]}", out_dir)
    print(out_name, "->", len(os.listdir(out_dir)), "фото")

make_subset(ABO_N_FRAMES, out_name="raw_dense")
make_subset(30)
make_subset(20)


raw_dense -> 72 фото
raw_N30 -> 30 фото
raw_N20 -> 20 фото


In [6]:
import json

def pose_spherical(theta_deg, phi_deg, radius):
    def trans_t(t):
        return np.array([[1,0,0,0],[0,1,0,0],[0,0,1,t],[0,0,0,1]], dtype=float)
    def rot_phi(p):
        p = np.deg2rad(p)
        return np.array([[1,0,0,0],[0,np.cos(p),-np.sin(p),0],[0,np.sin(p),np.cos(p),0],[0,0,0,1]], dtype=float)
    def rot_theta(t):
        t = np.deg2rad(t)
        return np.array([[np.cos(t),0,-np.sin(t),0],[0,1,0,0],[np.sin(t),0,np.cos(t),0],[0,0,0,1]], dtype=float)
    c2w = trans_t(radius)
    c2w = rot_phi(phi_deg) @ c2w
    c2w = rot_theta(theta_deg) @ c2w
    c2w = np.array([[-1,0,0,0],[0,0,1,0],[0,1,0,0],[0,0,0,1]], dtype=float) @ c2w
    return c2w

def build_synthetic_dataset(subset_dir_name, radius=2.5, elevation_deg=-15.0, fov_deg=40.0):
    raw_dir = f"{D_ABO}/{subset_dir_name}"
    names = sorted(os.listdir(raw_dir))
    w, h = Image.open(f"{raw_dir}/{names[0]}").size
    out_dir = f"{D_ABO}/{subset_dir_name}_synth"
    img_out = f"{out_dir}/images"
    os.makedirs(img_out, exist_ok=True)
    fov_x = np.deg2rad(fov_deg)
    fl = 0.5 * w / np.tan(0.5 * fov_x)
    frames = []
    for n in names:
        az = int(n.split('_')[1].split('.')[0])
        theta = az * 5.0
        c2w = pose_spherical(theta, elevation_deg, radius)
        dst = f"{img_out}/{n}"
        if not os.path.exists(dst):
            shutil.copy(f"{raw_dir}/{n}", dst)
        frames.append({"file_path": f"images/{n}", "transform_matrix": c2w.tolist()})
    out = {
        "fl_x": fl, "fl_y": fl, "cx": w / 2, "cy": h / 2, "w": w, "h": h,
        "camera_model": "OPENCV", "k1": 0.0, "k2": 0.0, "p1": 0.0, "p2": 0.0,
        "frames": frames,
    }
    with open(f"{out_dir}/transforms.json", "w") as f:
        json.dump(out, f, indent=2)
    print(subset_dir_name, "-> поз:", len(frames), "->", out_dir)

for name in ["raw_dense", "raw_N30", "raw_N20"]:
    build_synthetic_dataset(name)


raw_dense -> поз: 72 -> /content/abo_chair/raw_dense_synth
raw_N30 -> поз: 30 -> /content/abo_chair/raw_N30_synth
raw_N20 -> поз: 20 -> /content/abo_chair/raw_N20_synth


In [7]:
OBJECT_PROMPT_ABO = "a chair."

def make_masks(processed_dir, prompt=OBJECT_PROMPT_ABO):
    img_dir = f"{processed_dir}/images"
    mask_dir = f"{processed_dir}/masks"
    os.makedirs(mask_dir, exist_ok=True)
    names = sorted(os.listdir(img_dir))
    no_box = []
    for n in names:
        img = Image.open(f"{img_dir}/{n}").convert("RGB")
        W, H = img.size
        b, s = gdino_boxes(img, prompt, 0.3)
        if len(b) == 0:
            b, s = gdino_boxes(img, "furniture. object.", 0.22)
        keep = [bb for bb in b if 0.02 < ((bb[2]-bb[0])*(bb[3]-bb[1]))/(W*H) < 0.99]
        if not keep:
            no_box.append(n)
            mask = np.ones((H, W), bool)
        else:
            mask = sam_union(img, keep)
        Image.fromarray((mask*255).astype(np.uint8)).save(f"{mask_dir}/{os.path.splitext(n)[0]}.png")
    print(processed_dir, "-> масок:", len(os.listdir(mask_dir)), "| без бокса:", len(no_box), no_box[:5])

for name in ["raw_dense", "raw_N30", "raw_N20"]:
    make_masks(f"{D_ABO}/{name}_synth")


/content/abo_chair/raw_dense_synth -> масок: 72 | без бокса: 0 []
/content/abo_chair/raw_N30_synth -> масок: 30 | без бокса: 0 []
/content/abo_chair/raw_N20_synth -> масок: 20 | без бокса: 0 []


In [8]:
def add_mask_paths(out_name):
    d = f"{D_ABO}/{out_name}_synth"
    tf = f"{d}/transforms.json"
    data = json.load(open(tf))
    for fr in data["frames"]:
        fname = os.path.basename(fr["file_path"])
        stem = os.path.splitext(fname)[0]
        mask_file = f"masks/{stem}.png"
        if os.path.exists(f"{d}/{mask_file}"):
            fr["mask_path"] = mask_file
    json.dump(data, open(tf, "w"), indent=2)
    n_with_mask = sum(1 for fr in data["frames"] if "mask_path" in fr)
    print(out_name, "-> масок привязано:", n_with_mask, "/", len(data["frames"]))

for name in ["raw_dense", "raw_N30", "raw_N20"]:
    add_mask_paths(name)


raw_dense -> масок привязано: 72 / 72
raw_N30 -> масок привязано: 30 / 30
raw_N20 -> масок привязано: 20 / 20


In [9]:
import subprocess

def train_splatfacto(processed_dir, out_name, iters):
    cmd = (
        f"yes | TORCHDYNAMO_DISABLE=1 /content/nsenv/bin/ns-train splatfacto "
        f"--data {processed_dir} --max-num-iterations {iters} --steps-per-save 1000 "
        f"--vis tensorboard --viewer.quit-on-train-completion True "
        f"--output-dir {D_ABO}/outputs_{out_name} --pipeline.model.cull-alpha-thresh 0.1 "
        f"nerfstudio-data"
    )
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(out_name, "-> returncode", r.returncode)
    print(r.stdout[-1500:])
    if r.returncode:
        print(r.stderr[-1500:])

train_splatfacto(f"{D_ABO}/raw_dense_synth", "dense", iters=7000)
train_splatfacto(f"{D_ABO}/raw_N30_synth", "n30", iters=3000)
train_splatfacto(f"{D_ABO}/raw_N20_synth", "n20", iters=3000)


dense -> returncode 0
(99.57%)       43.956 ms            1 s, 318.677 ms      19.11 M               
6980 (99.71%)       44.175 ms            883.506 ms           19.02 M               
6990 (99.86%)       44.138 ms            441.379 ms           19.04 M               
6999 (99.99%)                                                                       
╭────────────────────────────────────────────── 🎉 Training Finished 🎉 ───────────────────────────────────────────────╮
│                        ╷                                                                                             │
│   Config File          │ /content/abo_chair/outputs_dense/raw_dense_synth/splatfacto/2026-09-30_191437/config.yml    │
│   Checkpoint Directory │ /content/abo_chair/outputs_dense/raw_dense_synth/splatfacto/2026-09-30_191437/nerfstudio…   │
│                        ╵                                                                                             │
╰───────────────────────────────────────

In [10]:
import glob

def export_and_mesh(out_name):
    cfgs = glob.glob(f"{D_ABO}/outputs_{out_name}/*/splatfacto/*/config.yml")
    if not cfgs:
        print(out_name, "-> config.yml не найден")
        return
    cfg = sorted(cfgs)[-1]
    export_dir = f"{D_ABO}/exports_{out_name}"
    subprocess.run(f"/content/nsenv/bin/ns-export gaussian-splat --load-config {cfg} --output-dir {export_dir}", shell=True)

    script = f'''
import numpy as np
import open3d as o3d
import trimesh

pcd = o3d.io.read_point_cloud("{export_dir}/splat.ply")
pcd, _ = pcd.remove_statistical_outlier(nb_neighbors=20, std_ratio=2.0)
nn = o3d.geometry.KDTreeFlann(pcd)
step = max(1, len(pcd.points) // 2000)
d = []
for i in range(0, len(pcd.points), step):
    _, idx, dist2 = nn.search_knn_vector_3d(pcd.points[i], 2)
    d.append(np.sqrt(dist2[1]))
eps = 5 * float(np.median(d))
labels = np.array(pcd.cluster_dbscan(eps=eps, min_points=10))
if labels.max() >= 0:
    biggest = np.bincount(labels[labels >= 0]).argmax()
    pcd = pcd.select_by_index(np.where(labels == biggest)[0])
pcd.estimate_normals(o3d.geometry.KDTreeSearchParamHybrid(radius=0.1, max_nn=30))
pcd.orient_normals_consistent_tangent_plane(30)
mesh, dens = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(pcd, depth=9)
dens = np.asarray(dens)
mesh.remove_vertices_by_mask(dens <= np.quantile(dens, 0.1))
mesh.remove_degenerate_triangles(); mesh.remove_duplicated_vertices()
mesh.remove_duplicated_triangles(); mesh.remove_non_manifold_edges()
mesh.compute_triangle_normals()
out = "{export_dir}/chair_object.stl"
o3d.io.write_triangle_mesh(out, mesh)
m = trimesh.load(out)
m.fill_holes(); m.fix_normals(); m.process(validate=True)
m.export(out)
print("STL:", out, "| Ecken:", len(m.vertices), "| Dreiecke:", len(m.faces))
print("watertight:", m.is_watertight)
'''
    open(f"/content/mesh_abo_{out_name}.py", "w").write(script)
    r = subprocess.run(["/content/nsenv/bin/python", f"/content/mesh_abo_{out_name}.py"], capture_output=True, text=True)
    print(r.stdout)
    if r.returncode:
        print(r.stderr[-2000:])

for name in ["dense", "n30", "n20"]:
    export_and_mesh(name)


STL: /content/abo_chair/exports_dense/chair_object.stl | Ecken: 289176 | Dreiecke: 562729
watertight: False

STL: /content/abo_chair/exports_n30/chair_object.stl | Ecken: 535585 | Dreiecke: 1034513
watertight: False

STL: /content/abo_chair/exports_n20/chair_object.stl | Ecken: 629149 | Dreiecke: 1212346
watertight: False



In [11]:
r = subprocess.run(["/content/nsenv/bin/pip", "install", "-q", "pygltflib"], capture_output=True, text=True)
print(r.returncode, r.stderr[-500:] if r.returncode else "pygltflib ok")


FileNotFoundError: [Errno 2] No such file or directory: '/content/nsenv/bin/pip'

In [12]:
r = subprocess.run(["uv", "pip", "install", "-q", "--python", "/content/nsenv/bin/python", "pygltflib"], capture_output=True, text=True)
print(r.returncode)
print(r.stdout[-500:])
if r.returncode:
    print(r.stderr[-1000:])


0



In [13]:
script = '''
import numpy as np
import open3d as o3d
import trimesh

def load_pts(path, n=30000):
    m = trimesh.load(path, force="mesh")
    pts, _ = trimesh.sample.sample_surface(m, n)
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(pts)
    return pcd

def normalize(pcd):
    pts = np.asarray(pcd.points)
    c = pts.mean(0)
    pts = pts - c
    scale = np.linalg.norm(pts, axis=1).max()
    pts = pts / scale
    out = o3d.geometry.PointCloud()
    out.points = o3d.utility.Vector3dVector(pts)
    return out

def chamfer_vs_ref(ref, cand_path, label):
    cand = normalize(load_pts(cand_path))
    best = None
    for deg in range(0, 360, 15):
        R = o3d.geometry.get_rotation_matrix_from_axis_angle([0, 0, np.deg2rad(deg)])
        c2 = o3d.geometry.PointCloud(cand)
        c2.rotate(R, center=(0, 0, 0))
        reg = o3d.pipelines.registration.registration_icp(
            c2, ref, 0.05, np.eye(4),
            o3d.pipelines.registration.TransformationEstimationPointToPoint())
        if best is None or reg.inlier_rmse < best[0]:
            best = (reg.inlier_rmse, deg, reg.transformation, R)
    rmse, deg, T, R0 = best
    aligned = o3d.geometry.PointCloud(cand)
    aligned.rotate(R0, center=(0, 0, 0))
    aligned.transform(T)
    reg2 = o3d.pipelines.registration.registration_icp(
        aligned, ref, 0.03, np.eye(4),
        o3d.pipelines.registration.TransformationEstimationPointToPoint())
    aligned.transform(reg2.transformation)
    d1 = np.asarray(aligned.compute_point_cloud_distance(ref))
    d2 = np.asarray(ref.compute_point_cloud_distance(aligned))
    chamfer = d1.mean() + d2.mean()
    print(label, ": best angle", deg, "deg, ICP inlier RMSE %.4f" % rmse, ", Chamfer %.4f" % chamfer)
    return chamfer

ref_cad = normalize(load_pts("/content/abo_chair/cad/B07F2X8K62.glb"))
for name in ["dense", "n30", "n20"]:
    path = "/content/abo_chair/exports_" + name + "/chair_object.stl"
    import os as _os
    if _os.path.exists(path):
        chamfer_vs_ref(ref_cad, path, name + " vs. ABO-CAD")
'''
open('/content/chamfer_vs_cad.py', 'w').write(script)
r = subprocess.run(['/content/nsenv/bin/python', '/content/chamfer_vs_cad.py'], capture_output=True, text=True)
print(r.stdout)
if r.returncode:
    print('STDERR:', r.stderr[-3000:])


dense vs. ABO-CAD : best angle 345 deg, ICP inlier RMSE 0.0273 , Chamfer 0.2236
n30 vs. ABO-CAD : best angle 270 deg, ICP inlier RMSE 0.0279 , Chamfer 0.2355
n20 vs. ABO-CAD : best angle 75 deg, ICP inlier RMSE 0.0283 , Chamfer 0.2132



In [14]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [1]:
from google.colab import files
files.download('/content/abo_chair/exports_dense/chair_object.stl')


FileNotFoundError: Cannot find file: /content/abo_chair/exports_dense/chair_object.stl

In [2]:
import os
print("abo_chair:", os.path.exists('/content/abo_chair'))
print("exports_dense dir:", os.listdir('/content/abo_chair') if os.path.exists('/content/abo_chair') else "нет abo_chair")


abo_chair: False
exports_dense dir: нет abo_chair
